In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

EVAL_ROOT = Path("../runs/eval_sa2_large")
CHECKPOINTS = sorted([p for p in EVAL_ROOT.iterdir() if p.is_dir()])
LABELS      = [p.name for p in CHECKPOINTS]
COLORS      = ["#4C72B0", "#DD8452", "#55A868"]
print("Checkpoints found:", LABELS)

def load_checkpoint(p: Path) -> dict:
    m = p / "metrics"
    d = {}
    for fname, cols in [
        ("spectral.csv",   ["si_sdr", "sdr", "stft_loss"]),
        ("cdpam.csv",      ["cdpam"]),
    ]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            for c in cols:
                if c in df.columns:
                    vals = df[c].dropna().values.astype(float)
                    d[c] = vals[np.isfinite(vals)]
    # CLAP — new format: separate clap_music.csv / clap_audio.csv (cosine column)
    for key, fname in [("clap_music", "clap_music.csv"), ("clap_audio", "clap_audio.csv")]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            if "cosine" in df.columns:
                vals = df["cosine"].dropna().values.astype(float)
                d[key] = vals[np.isfinite(vals)]
    # CLAP — old format: single clap_score.csv (clap_music / clap_audio columns)
    if (m / "clap_score.csv").exists() and ("clap_music" not in d or "clap_audio" not in d):
        df = pd.read_csv(m / "clap_score.csv")
        for c in ["clap_music", "clap_audio"]:
            if c in df.columns and c not in d:
                vals = df[c].dropna().values.astype(float)
                d[c] = vals[np.isfinite(vals)]
    for fname, key in [("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud")]:
        if (m / fname).exists():
            d[key] = float(pd.read_csv(m / fname).iloc[0]["score"])
    return d


data = {lbl: load_checkpoint(p) for lbl, p in zip(LABELS, CHECKPOINTS)}

PER_FILE      = ["si_sdr", "sdr", "stft_loss", "clap_music", "clap_audio", "cdpam"]
SCALAR        = ["fad_mert", "fad_gudgud"]
HIGHER_BETTER = {"si_sdr", "sdr", "clap_music", "clap_audio"}

def highlight_best(s):
    finite = s[s.notna()]
    if finite.empty:
        return [""] * len(s)
    best = finite.max() if s.name in HIGHER_BETTER else finite.min()
    return ["font-weight:bold; color:#00cc66" if (np.isfinite(v) and v == best) else "" for v in s]

Checkpoints found: ['sa2-step=100000', 'sa2-step=250000', 'sa2-step=500000', 'sa2Disc-step=100000', 'sa2Disc-step=250000', 'sa2Disc-step=500000']


In [1]:

rows = []
for lbl, d in data.items():
    row = {"checkpoint": lbl}
    for k in PER_FILE:
        row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
    for k in SCALAR:
        row[k] = d.get(k, float("nan"))
    rows.append(row)

summary = pd.DataFrame(rows).set_index("checkpoint")

n = {k: max([len(d.get(k, [])) for d in data.values()]) for k in PER_FILE}
n_str = f"{n['si_sdr']} files" if len(set(v for v in n.values() if v > 0)) == 1 else str({k: v for k, v in n.items() if v > 0})

print(f"\n  FINAL TEST RESULTS — FMA-large  ({n_str} per checkpoint)\n")
display(summary.style
        .apply(highlight_best)
        .format({**{k: "{:.4f}" for k in PER_FILE_SWIN + SCALAR_SWIN}, "cdpam": "{:.3f}"}, na_rep="—"))


NameError: name 'data' is not defined

In [4]:
# ── Swin C-VAE Results ──────────────────────────────────────────
EVAL_ROOT_SWIN = Path("../runs/eval_swin_large")  # → eval_swin_large per run completa

PER_FILE_SWIN      = ["si_sdr", "sdr", "stft_loss", "clap_music", "clap_audio", "cdpam"]
SCALAR_SWIN        = ["fad_mert", "fad_gudgud"]
HIGHER_BETTER_SWIN = {"si_sdr", "sdr", "clap_music", "clap_audio"}

def highlight_best_swin(s):
    finite = s[s.notna()]
    if finite.empty:
        return [""] * len(s)
    best = finite.max() if s.name in HIGHER_BETTER_SWIN else finite.min()
    return ["font-weight:bold; color:#00cc66" if (np.isfinite(v) and v == best) else "" for v in s]

if EVAL_ROOT_SWIN.exists():
    CHECKPOINTS_SWIN = sorted([p for p in EVAL_ROOT_SWIN.iterdir() if p.is_dir()])
    LABELS_SWIN      = [p.name for p in CHECKPOINTS_SWIN]
    print("Swin checkpoints found:", LABELS_SWIN)

    data_swin = {lbl: load_checkpoint(p) for lbl, p in zip(LABELS_SWIN, CHECKPOINTS_SWIN)}

    rows_swin = []
    for lbl, d in data_swin.items():
        row = {"checkpoint": lbl}
        for k in PER_FILE_SWIN:
            row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
        for k in SCALAR_SWIN:
            row[k] = d.get(k, float("nan"))
        rows_swin.append(row)

    summary_swin = pd.DataFrame(rows_swin).set_index("checkpoint")

    n_swin = len(data_swin[LABELS_SWIN[0]].get("si_sdr", []))
    print(f"\n  SWIN VarT RESULTS  ({n_swin} files per checkpoint)\n")
    score = (summary_swin["fad_mert"] + summary_swin["fad_gudgud"] + summary_swin["cdpam"])
    summary_swin = summary_swin.assign(composite_score=score)
    summary_swin = summary_swin.sort_values("composite_score", ascending=True)
    display(summary_swin.style
            .apply(highlight_best_swin)
            .format({**{k: "{:.4f}" for k in PER_FILE_SWIN + SCALAR_SWIN}, "cdpam": "{:.3f}"}, na_rep="—"))
else:
    print(f"Swin evaluation root not found at {EVAL_ROOT_SWIN}")

Swin checkpoints found: ['COOLDOWN_LARGE2_mr05_e49', 'FT1_real_decFT_sao_disc', 'FT2_discRiparte', 'FT2_discRiprende', 'FT3_O4b_decFT_wavtok', 'FT4_O4b_clap_decFT_wavtok', 'FT5_O8_decFT_wavtok', 'FT5_det', 'FT7_O8_decFT_postnet', 'FT8_FT5_cooldown', 'FT8_deterministic', 'J4_ps64x1_3s_e256_l1_mrmel', 'LARGE1_ACE_e107_LIVE', 'LARGE1_e41_331k', 'LARGE1_e89_470k', 'LARGE2_fold_last', 'LARGE2_mrmel05_e38_308k', 'LARGE2_mrmel05_e99', 'M11_real_softnorm', 'MED_C_fold_last', 'MED_P3_x128_last', 'MSE_ps32x1_4s_e128', 'N11_real_mse_bigvgan_fm05', 'N11_real_mse_bigvgan_fm1_lr1e-3', 'N12_real_mse_hil_rpgan', 'N2_real_mse_ifgf', 'N3_complex_mse_ifgf', 'N4_real_jepa', 'N5_real_perceptual_mse', 'N6_real_mse_sao_disc', 'N7_real_mse_sao_rpgan', 'N8_real_mse_sao_disc_psycho', 'Nwavtok_original', 'O1_real_same_mrstft', 'O1_wavTokenizer', 'O2_real_flow_dit', 'O3_mrmel', 'O3_real_vf_mert', 'O3b_real_vf_mert_tight', 'O4b_real_distill_clap', 'O4b_real_distill_mert', 'O4c_real_distill_clap_tiny', 'O5_real_chr

,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud,composite_score
checkpoint,,,,,,,,,
epoch_339,2.2294,4.0584,1.0142,0.9483,0.9508,0.091,0.1929,0.0173,0.301587
epoch_299,2.2233,4.0554,1.0098,0.9481,0.9505,0.092,0.1948,0.0172,0.303800
COOLDOWN_LARGE2_mr05_e49,2.2200,4.0544,1.0101,0.9485,0.9510,0.091,0.1960,0.0170,0.304401
epoch_259,2.2193,4.0578,1.0113,0.9481,0.9503,0.092,0.1962,0.0174,0.305708
epoch_199,2.1966,4.0516,1.0078,0.9474,0.9496,0.093,0.1984,0.0179,0.309307
LARGE2_mrmel05_e99,2.1178,4.0760,1.0330,0.9394,0.9432,0.086,0.2309,0.0225,0.339627
FT1_real_decFT_sao_disc,2.4883,5.0009,1.0502,0.8188,0.8125,0.144,0.5475,—,—
FT2_discRiparte,-1.9787,2.5375,1.0753,0.7298,0.7005,0.216,0.9587,—,—
FT2_discRiprende,-7.8666,0.5215,1.7334,0.4725,0.4654,0.490,2.5530,—,—


In [ ]:
# ── MAEB (audio-only) Results — TRANSPOSED (models × tasks) ──────────────────
# Multi-model: ogni subdir sotto MAEB_ROOT è un checkpoint separato.
# Righe = modelli, colonne = task (raggruppate per Type). Tutte higher-is-better.
# Esclusioni/rename modelli sono fatti SUL DISCO (dir runs/maeb_*).
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

MAEB_ROOTS = [Path("../runs/maeb_sao"), Path("../runs/maeb_swin")]
EXCLUDE_TASKS = {"NMSQAPairClassification", "MridinghamTonic", "NSynth", "GTZANGenreClustering"}   # speech, saturo ~0.49 per tutti → fuori dominio

TASK_TYPE = {
    "JamAltArtistA2ARetrieval":          "Retrieval",
    "FMAArtistA2ARetrieval":             "Retrieval",
    "GTZANAudioReranking":               "Reranking",
    "FMAGenreAudioReranking":            "Reranking",
    "BeijingOpera":                       "Classification",
    "BirdCLEF":                           "Classification",
    "CREMA_D":                            "Classification",
    "CommonLanguageAgeDetection":         "Classification",
    "GTZANGenre":                         "Classification",
    "FMAGenreClassification":             "Classification",
    "IEMOCAPGender":                      "Classification",
    "MInDS14":                            "Classification",
    "MridinghamTonic":                    "Classification",
    "MridinghamStroke":                   "Classification",
    "NSynth":                             "Classification",
    "SIBFLEURS":                          "Classification",
    "VoxCelebSA":                         "Classification",
    "VoxPopuliLanguageID":                "Classification",
    "CREMA_DClustering":                  "Clustering",
    "GTZANGenreClustering":               "Clustering",
    "MusicGenreClustering":               "Clustering",
    "FMAGenreClustering":                 "Clustering",
    "FMAArtistClustering":                "Clustering",
    "VehicleSoundClustering":             "Clustering",
    "VoxPopuliGenderClustering":          "Clustering",
    "CREMADPairClassification":           "PairClassification",
    "NMSQAPairClassification":            "PairClassification",
    "FMAArtistPairClassification":        "PairClassification",
    "VoxPopuliAccentPairClassification":  "PairClassification",
}
TYPE_METRIC = {
    "Retrieval":          "ndcg_at_10",
    "Reranking":          "map",
    "Classification":     "accuracy",
    "Clustering":         "v_measure",
    "PairClassification": "similarity_ap",
}
TYPE_ORDER = ["Classification", "Clustering", "Retrieval", "Reranking", "PairClassification"]

def _main_score_from_json(path: Path) -> float | None:
    d = json.loads(path.read_text())
    for split_scores in d.get("scores", {}).values():
        if isinstance(split_scores, list):
            for s in split_scores:
                if "main_score" in s:
                    return float(s["main_score"])
        elif isinstance(split_scores, dict) and "main_score" in split_scores:
            return float(split_scores["main_score"])
    return None

def _load_model(local_dir: Path) -> tuple[dict[str, float | None], dict]:
    """Load per-task scores + metadata from a model's local/ dir."""
    results: dict[str, float | None] = {}
    for p in sorted(local_dir.glob("*.json")):
        if p.stem not in ("summary", "model_meta"):
            results[p.stem] = _main_score_from_json(p)
    meta = {}
    sp = local_dir / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        for k, v in s.get("results", {}).items():
            results[k] = v
        meta = {k: s[k] for k in ("ckpt", "embed_dim", "pooling", "is_complex") if k in s}
    return results, meta

def _short_label(local_dir: Path) -> str:
    name = local_dir.parent.name
    for prefix in ("sao-ace__", "cvae-cplx__", "cvae-real__"):
        name = name.replace(prefix, "")
    return name

# ── discover model subdirs ────────────────────────────────────────────────────
model_local_dirs = []
for root in MAEB_ROOTS:
    if root.exists():
        model_local_dirs.extend(
            p / "local" for p in root.iterdir()
            if p.is_dir() and (p / "local").is_dir()
        )
model_local_dirs = sorted(model_local_dirs, key=lambda d: d.parent.name)

if not model_local_dirs:
    print(f"No results found in {[str(r) for r in MAEB_ROOTS]}")
else:
    labels = [_short_label(d) for d in model_local_dirs]
    all_results, all_meta = {}, {}
    for lbl, d in zip(labels, model_local_dirs):
        all_results[lbl], all_meta[lbl] = _load_model(d)

    print(f"MAEB  ·  {len(labels)} checkpoint(s)\n")
    for lbl in labels:
        n_done = sum(1 for k, v in all_results[lbl].items() if v is not None and k not in EXCLUDE_TASKS)
        meta_str = "  ".join(f"{k}={v}" for k, v in all_meta[lbl].items())
        print(f"  {lbl:<40} {n_done:>2} tasks   {meta_str}")
    print()

    # ── task ordering: core tasks (FMA suite) first, then extra tasks ────────
    CORE_TASKS = [
        "FMAGenreClassification",
        "FMAGenreClustering",
        "FMAArtistClustering",
        "FMAArtistA2ARetrieval",
        "FMAGenreAudioReranking",
        "FMAArtistPairClassification",
    ]
    CORE_SET = set(CORE_TASKS)

    all_tasks = sorted({t for r in all_results.values() for t in r} - EXCLUDE_TASKS)
    by_type: dict[str, list[str]] = defaultdict(list)
    for t in all_tasks:
        by_type[TASK_TYPE.get(t, "Other")].append(t)
    
    raw_ordered_tasks = []
    for ttype in TYPE_ORDER + [k for k in by_type if k not in TYPE_ORDER]:
        for task in sorted(by_type.get(ttype, [])):
            raw_ordered_tasks.append(task)
            
    # Partition: core tasks first, followed by extra tasks
    core_ordered = [t for t in raw_ordered_tasks if t in CORE_SET]
    extra_ordered = [t for t in raw_ordered_tasks if t not in CORE_SET]
    ordered_tasks = core_ordered + extra_ordered
    
    col_tuples = []
    for task in ordered_tasks:
        col_tuples.append((TASK_TYPE.get(task, "Other"), task))

    # ── transposed DF: rows = models, columns = (Type, Task) ──────────────────
    cols = pd.MultiIndex.from_tuples(col_tuples + [("", "AVG")], names=["Type", "Task"])
    rows = []
    for lbl in labels:
        vals = [all_results[lbl].get(t) for t in ordered_tasks]
        # Consider only core tasks for AVG and divide by the total number of core columns (6)
        core_vals = [all_results[lbl].get(t) for t in ordered_tasks if t in CORE_SET]
        core_finite = [v for v in core_vals if v is not None]
        avg_val = sum(core_finite) / len(CORE_TASKS) if CORE_TASKS else float("nan")
        vals.append(avg_val)
        rows.append(vals)
    df_maeb = pd.DataFrame(rows, index=labels, columns=cols)
    df_maeb.index.name = "checkpoint"

    # ── sort: decrescente sulla colonna AVG ──────────────────────────────────
    _avg = pd.to_numeric(df_maeb[("", "AVG")], errors="coerce")
    df_maeb = df_maeb.loc[_avg.sort_values(ascending=False, na_position="last").index]

    # highlight best per task column (all higher-is-better)
    def _hl_best_col(s):
        finite = s[s.notna()]
        if finite.empty:
            return [""] * len(s)
        best = finite.max()
        return ["font-weight:bold; color:#4C72B0" if (pd.notna(v) and v == best) else "" for v in s]

    print("Metriche per Type:", TYPE_METRIC)
    display(
        df_maeb.style
        .apply(_hl_best_col, axis=0)
        .format("{:.4f}", na_rep="—")
    )


# ── discover model subdirs ────────────────────────────────────────────────────
model_local_dirs = []
for root in MAEB_ROOTS:
    if root.exists():
        model_local_dirs.extend(
            p / "local" for p in root.iterdir()
            if p.is_dir() and (p / "local").is_dir()
        )
model_local_dirs = sorted(model_local_dirs, key=lambda d: d.parent.name)

if not model_local_dirs:
    print(f"No results found in {[str(r) for r in MAEB_ROOTS]}")
else:
    labels = [_short_label(d) for d in model_local_dirs]
    all_results, all_meta = {}, {}
    for lbl, d in zip(labels, model_local_dirs):
        all_results[lbl], all_meta[lbl] = _load_model(d)

    print(f"MAEB  ·  {len(labels)} checkpoint(s)\n")
    for lbl in labels:
        n_done = sum(1 for k, v in all_results[lbl].items() if v is not None and k not in EXCLUDE_TASKS)
        meta_str = "  ".join(f"{k}={v}" for k, v in all_meta[lbl].items())
        print(f"  {lbl:<40} {n_done:>2} tasks   {meta_str}")
    print()

    # ── task ordering: core tasks (FMA suite) first, then extra tasks ────────
    CORE_TASKS = [
        "FMAGenreClassification",
        "FMAGenreClustering",
        "FMAArtistClustering",
        "FMAArtistA2ARetrieval",
        "FMAGenreAudioReranking",
        "FMAArtistPairClassification",
    ]
    CORE_SET = set(CORE_TASKS)

    all_tasks = sorted({t for r in all_results.values() for t in r} - EXCLUDE_TASKS)
    by_type: dict[str, list[str]] = defaultdict(list)
    for t in all_tasks:
        by_type[TASK_TYPE.get(t, "Other")].append(t)
    
    raw_ordered_tasks = []
    for ttype in TYPE_ORDER + [k for k in by_type if k not in TYPE_ORDER]:
        for task in sorted(by_type.get(ttype, [])):
            raw_ordered_tasks.append(task)
            
    # Partition: core tasks first, followed by extra tasks
    core_ordered = [t for t in raw_ordered_tasks if t in CORE_SET]
    extra_ordered = [t for t in raw_ordered_tasks if t not in CORE_SET]
    ordered_tasks = core_ordered + extra_ordered
    
    col_tuples = []
    for task in ordered_tasks:
        col_tuples.append((TASK_TYPE.get(task, "Other"), task))

    # ── transposed DF: rows = models, columns = (Type, Task) ──────────────────
    cols = pd.MultiIndex.from_tuples(col_tuples + [("", "AVG")], names=["Type", "Task"])
    rows = []
    for lbl in labels:
        vals = [all_results[lbl].get(t) for t in ordered_tasks]
        # Consider only core tasks for AVG and divide by the total number of core columns (6)
        core_vals = [all_results[lbl].get(t) for t in ordered_tasks if t in CORE_SET]
        core_finite = [v for v in core_vals if v is not None]
        avg_val = sum(core_finite) / len(CORE_TASKS) if CORE_TASKS else float("nan")
        vals.append(avg_val)
        rows.append(vals)
    df_maeb = pd.DataFrame(rows, index=labels, columns=cols)
    df_maeb.index.name = "checkpoint"

    # ── sort: decrescente sulla colonna AVG ──────────────────────────────────
    _avg = pd.to_numeric(df_maeb[("", "AVG")], errors="coerce")
    df_maeb = df_maeb.loc[_avg.sort_values(ascending=False, na_position="last").index]

    # highlight best per task column (all higher-is-better)
    def _hl_best_col(s):
        finite = s[s.notna()]
        if finite.empty:
            return [""] * len(s)
        best = finite.max()
        return ["font-weight:bold; color:#4C72B0" if (pd.notna(v) and v == best) else "" for v in s]

    print("Metriche per Type:", TYPE_METRIC)
    display(
        df_maeb.style
        .apply(_hl_best_col, axis=0)
        .format("{:.4f}", na_rep="—")
    )

MAEB  ·  33 checkpoint(s)

  baseline_MSE_prec32__std                  6 tasks   ckpt=checkpoints/baseline_MSE_prec32.ckpt  embed_dim=64  is_complex=True
  FT1_real_decFT_sao_disc__std              6 tasks   ckpt=checkpoints/FT1_real_decFT_sao_disc.ckpt  embed_dim=64  is_complex=False
  FT2_discRiparte__std                      6 tasks   ckpt=checkpoints/FT2_discRiparte.ckpt  embed_dim=64  is_complex=False
  FT2_discRiprende__std                     6 tasks   ckpt=checkpoints/FT2_discRiprende.ckpt  embed_dim=64  is_complex=False
  FT3_O4b_decFT_wavtok__std                 6 tasks   ckpt=checkpoints/FT3_O4b_decFT_wavtok.ckpt  embed_dim=64  is_complex=False
  FT4_O4b_clap_decFT_wavtok__std            8 tasks   ckpt=checkpoints/FT4_O4b_clap_decFT_wavtok.ckpt  embed_dim=64  is_complex=False
  FT5_O8_decFT_wavtok__std                  8 tasks   ckpt=checkpoints/FT5_O8_decFT_wavtok.ckpt  embed_dim=64  is_complex=False
  FT7_O8_decFT_postnet__std                 8 tasks   ckpt=checkpoints/FT7

In [6]:
# ── MAEB (audio-only) Results — TRANSPOSED (models × tasks) ──────────────────
# Multi-model: ogni subdir sotto MAEB_ROOT è un checkpoint separato.
# Righe = modelli, colonne = task (raggruppate per Type). Tutte higher-is-better.
# Esclusioni/rename modelli sono fatti SUL DISCO (dir runs/maeb_*).
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

MAEB_ROOTS = [Path("../runs/maeb_sao"), Path("../runs/maeb_swin")]
EXCLUDE_TASKS = {"NMSQAPairClassification", "MridinghamTonic", "NSynth", "GTZANGenreClustering"}   # speech, saturo ~0.49 per tutti → fuori dominio

TASK_TYPE = {
    "JamAltArtistA2ARetrieval":          "Retrieval",
    "FMAArtistA2ARetrieval":             "Retrieval",
    "GTZANAudioReranking":               "Reranking",
    "FMAGenreAudioReranking":            "Reranking",
    "BeijingOpera":                       "Classification",
    "BirdCLEF":                           "Classification",
    "CREMA_D":                            "Classification",
    "CommonLanguageAgeDetection":         "Classification",
    "GTZANGenre":                         "Classification",
    "FMAGenreClassification":             "Classification",
    "IEMOCAPGender":                      "Classification",
    "MInDS14":                            "Classification",
    "MridinghamTonic":                    "Classification",
    "MridinghamStroke":                   "Classification",
    "NSynth":                             "Classification",
    "SIBFLEURS":                          "Classification",
    "VoxCelebSA":                         "Classification",
    "VoxPopuliLanguageID":                "Classification",
    "CREMA_DClustering":                  "Clustering",
    "GTZANGenreClustering":               "Clustering",
    "MusicGenreClustering":               "Clustering",
    "FMAGenreClustering":                 "Clustering",
    "FMAArtistClustering":                "Clustering",
    "VehicleSoundClustering":             "Clustering",
    "VoxPopuliGenderClustering":          "Clustering",
    "CREMADPairClassification":           "PairClassification",
    "NMSQAPairClassification":            "PairClassification",
    "FMAArtistPairClassification":        "PairClassification",
    "VoxPopuliAccentPairClassification":  "PairClassification",
}
TYPE_METRIC = {
    "Retrieval":          "ndcg_at_10",
    "Reranking":          "map",
    "Classification":     "accuracy",
    "Clustering":         "v_measure",
    "PairClassification": "similarity_ap",
}
TYPE_ORDER = ["Classification", "Clustering", "Retrieval", "Reranking", "PairClassification"]

def _main_score_from_json(path: Path) -> float | None:
    d = json.loads(path.read_text())
    for split_scores in d.get("scores", {}).values():
        if isinstance(split_scores, list):
            for s in split_scores:
                if "main_score" in s:
                    return float(s["main_score"])
        elif isinstance(split_scores, dict) and "main_score" in split_scores:
            return float(split_scores["main_score"])
    return None

def _load_model(local_dir: Path) -> tuple[dict[str, float | None], dict]:
    """Load per-task scores + metadata from a model's local/ dir."""
    results: dict[str, float | None] = {}
    for p in sorted(local_dir.glob("*.json")):
        if p.stem not in ("summary", "model_meta"):
            results[p.stem] = _main_score_from_json(p)
    meta = {}
    sp = local_dir / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        for k, v in s.get("results", {}).items():
            results[k] = v
        meta = {k: s[k] for k in ("ckpt", "embed_dim", "pooling", "is_complex") if k in s}
    return results, meta

def _short_label(local_dir: Path) -> str:
    name = local_dir.parent.name
    for prefix in ("sao-ace__", "cvae-cplx__", "cvae-real__"):
        name = name.replace(prefix, "")
    return name

# ── discover model subdirs ────────────────────────────────────────────────────
model_local_dirs = []
for root in MAEB_ROOTS:
    if root.exists():
        model_local_dirs.extend(
            p / "local" for p in root.iterdir()
            if p.is_dir() and (p / "local").is_dir()
        )
model_local_dirs = sorted(model_local_dirs, key=lambda d: d.parent.name)

if not model_local_dirs:
    print(f"No results found in {[str(r) for r in MAEB_ROOTS]}")
else:
    labels = [_short_label(d) for d in model_local_dirs]
    all_results, all_meta = {}, {}
    for lbl, d in zip(labels, model_local_dirs):
        all_results[lbl], all_meta[lbl] = _load_model(d)

    print(f"MAEB  ·  {len(labels)} checkpoint(s)\n")
    for lbl in labels:
        n_done = sum(1 for k, v in all_results[lbl].items() if v is not None and k not in EXCLUDE_TASKS)
        meta_str = "  ".join(f"{k}={v}" for k, v in all_meta[lbl].items())
        print(f"  {lbl:<40} {n_done:>2} tasks   {meta_str}")
    print()

    # ── task ordering: core tasks (FMA suite) first, then extra tasks ────────
    CORE_TASKS = [
        "FMAGenreClassification",
        "FMAGenreClustering",
        "FMAArtistClustering",
        "FMAArtistA2ARetrieval",
        "FMAGenreAudioReranking",
        "FMAArtistPairClassification",
    ]
    CORE_SET = set(CORE_TASKS)

    all_tasks = sorted({t for r in all_results.values() for t in r} - EXCLUDE_TASKS)
    by_type: dict[str, list[str]] = defaultdict(list)
    for t in all_tasks:
        by_type[TASK_TYPE.get(t, "Other")].append(t)
    
    raw_ordered_tasks = []
    for ttype in TYPE_ORDER + [k for k in by_type if k not in TYPE_ORDER]:
        for task in sorted(by_type.get(ttype, [])):
            raw_ordered_tasks.append(task)
            
    # Partition: core tasks first, followed by extra tasks
    core_ordered = [t for t in raw_ordered_tasks if t in CORE_SET]
    extra_ordered = [t for t in raw_ordered_tasks if t not in CORE_SET]
    ordered_tasks = core_ordered + extra_ordered
    
    col_tuples = []
    for task in ordered_tasks:
        col_tuples.append((TASK_TYPE.get(task, "Other"), task))

    # ── transposed DF: rows = models, columns = (Type, Task) ──────────────────
    cols = pd.MultiIndex.from_tuples(col_tuples + [("", "AVG")], names=["Type", "Task"])
    rows = []
    for lbl in labels:
        vals = [all_results[lbl].get(t) for t in ordered_tasks]
        # Consider only core tasks for AVG and divide by the total number of core columns (6)
        core_vals = [all_results[lbl].get(t) for t in ordered_tasks if t in CORE_SET]
        core_finite = [v for v in core_vals if v is not None]
        avg_val = sum(core_finite) / len(CORE_TASKS) if CORE_TASKS else float("nan")
        vals.append(avg_val)
        rows.append(vals)
    df_maeb = pd.DataFrame(rows, index=labels, columns=cols)
    df_maeb.index.name = "checkpoint"

    # ── sort: decrescente sulla colonna AVG ──────────────────────────────────
    _avg = pd.to_numeric(df_maeb[("", "AVG")], errors="coerce")
    df_maeb = df_maeb.loc[_avg.sort_values(ascending=False, na_position="last").index]

    # highlight best per task column (all higher-is-better)
    def _hl_best_col(s):
        finite = s[s.notna()]
        if finite.empty:
            return [""] * len(s)
        best = finite.max()
        return ["font-weight:bold; color:#4C72B0" if (pd.notna(v) and v == best) else "" for v in s]

    print("Metriche per Type:", TYPE_METRIC)
    display(
        df_maeb.style
        .apply(_hl_best_col, axis=0)
        .format("{:.4f}", na_rep="—")
    )

MAEB  ·  11 checkpoint(s)

  LARGE1_ACE_e107_LIVE__std                 8 tasks   ckpt=checkpoints/LARGE1_ACE_e107_LIVE.ckpt  embed_dim=64  is_complex=False
  LARGE1_e41_331k__std                      6 tasks   ckpt=checkpoints/LARGE1_e41_331k.ckpt  embed_dim=64  is_complex=False
  LARGE1_e89_470k__std                      6 tasks   ckpt=checkpoints/LARGE1_e89_470k.ckpt  embed_dim=64  is_complex=False
  LARGE2_fold_last__std                     8 tasks   ckpt=checkpoints/LARGE2_fold_last.ckpt  embed_dim=64  is_complex=False
  LARGE2_mrmel05_e38_308k__std              6 tasks   ckpt=checkpoints/LARGE2_mrmel05_e38_308k.ckpt  embed_dim=64  is_complex=False
  LARGE2_mrmel05_e99__std                  15 tasks   ckpt=/leonardo/home/userexternal/fbrigant/work/C-VAE/checkpoints/LARGE2_mrmel05_e99.ckpt  embed_dim=64  is_complex=False
  MED_C_fold_last__std                      8 tasks   ckpt=checkpoints/MED_C_fold_last.ckpt  embed_dim=64  is_complex=False
  O8_EMA__std                           

# SOTA

In [4]:
# ════════════════════════════════════════════════════════════════════════════
# SOTA BASELINES — reconstruction (runs/eval_sota_large) + MAEB (runs/maeb_<model>)
# SELF-CONTAINED: run ONLY this cell (no need for cells 0 or 5).
# codicodec · music2latent · SAME · sao-vae — full FMA-large test, same metric
# pipeline + shared FAD target cache as Swin/SAO → directly comparable.
# (CSVs/JSONs appear as jobs finish; not-yet-computed metrics show as "—".)
# ════════════════════════════════════════════════════════════════════════════
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

# ── recon loader (self-contained copy of cell 0's load_checkpoint) ───────────
def _load_recon(p: Path) -> dict:
    m, d = p / "metrics", {}
    for fname, cols in [("spectral.csv", ["si_sdr", "sdr", "stft_loss"]), ("cdpam.csv", ["cdpam"])]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            for c in cols:
                if c in df.columns:
                    v = df[c].dropna().values.astype(float); d[c] = v[np.isfinite(v)]
    for key, fname in [("clap_music", "clap_music.csv"), ("clap_audio", "clap_audio.csv")]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            if "cosine" in df.columns:
                v = df["cosine"].dropna().values.astype(float); d[key] = v[np.isfinite(v)]
    for fname, key in [("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud")]:
        if (m / fname).exists():
            d[key] = float(pd.read_csv(m / fname).iloc[0]["score"])
    return d

EVAL_ROOT_SOTA     = Path("../runs/eval_sota_large")
RECON_MODELS_SOTA  = ["codicodec", "music2latent", "same", "sao-vae"]      # fixed rows (— until ready)
PER_FILE_SOTA      = ["si_sdr", "sdr", "stft_loss", "clap_music", "clap_audio", "cdpam"]
SCALAR_SOTA        = ["fad_mert", "fad_gudgud"]
HIGHER_BETTER_SOTA = {"si_sdr", "sdr", "clap_music", "clap_audio"}

def _hl_sota(s):
    finite = s[s.notna()]
    if finite.empty:
        return [""] * len(s)
    best = finite.max() if s.name in HIGHER_BETTER_SOTA else finite.min()
    return ["font-weight:bold; color:#00cc66" if (np.isfinite(v) and v == best) else "" for v in s]

rows = []
for name in RECON_MODELS_SOTA:
    d = _load_recon(EVAL_ROOT_SOTA / name) if (EVAL_ROOT_SOTA / name / "metrics").exists() else {}
    row = {"model": name}
    for k in PER_FILE_SOTA:
        row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
    for k in SCALAR_SOTA:
        row[k] = d.get(k, float("nan"))
    row["n_files"] = int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
    rows.append(row)
summary_sota = pd.DataFrame(rows).set_index("model")
print("  SOTA BASELINES — reconstruction (FMA-large test)\n")
display(summary_sota.style
        .apply(_hl_sota, subset=PER_FILE_SOTA + SCALAR_SOTA)
        .format({**{k: "{:.4f}" for k in PER_FILE_SOTA + SCALAR_SOTA},
                 "cdpam": "{:.3f}", "n_files": "{:.0f}"}, na_rep="—"))

# ── MAEB (semantic probing, 64-d; SAME excluded → 256-d) ─────────────────────
# Columns built from the FULL expected task set (core FMA + extra) in fixed
# Type-order (core-first) so every task always shows, with "—" until computed.
MAEB_ROOTS_SOTA = {"codicodec": Path("../runs/maeb_codicodec"),
                   "music2latent": Path("../runs/maeb_music2latent"),
                   "sao-vae": Path("../runs/maeb_sao-vae"),
                    "same": Path("../runs/maeb_same")}
FMA_SUITE_SOTA  = ["FMAGenreClassification", "FMAGenreClustering", "FMAArtistClustering",
                   "FMAArtistA2ARetrieval", "FMAGenreAudioReranking", "FMAArtistPairClassification"]
EXTRA_SOTA      = ["GTZANGenre", "MusicGenreClustering"]
TASK_TYPE_SOTA  = {
    "FMAGenreClassification": "Classification", "GTZANGenre": "Classification",
    "FMAGenreClustering": "Clustering", "FMAArtistClustering": "Clustering",
    "MusicGenreClustering": "Clustering",
    "FMAArtistA2ARetrieval": "Retrieval", "FMAGenreAudioReranking": "Reranking",
    "FMAArtistPairClassification": "PairClassification",
}
TYPE_ORDER_SOTA = ["Classification", "Clustering", "Retrieval", "Reranking", "PairClassification"]
_CORE_SET = set(FMA_SUITE_SOTA)

def _main_score(path: Path):
    d = json.loads(path.read_text())
    for sp in d.get("scores", {}).values():
        if isinstance(sp, list):
            for s in sp:
                if "main_score" in s:
                    return float(s["main_score"])
        elif isinstance(sp, dict) and "main_score" in sp:
            return float(sp["main_score"])
    return None

def _load_maeb(root: Path) -> dict:
    res = {}
    if not root.exists():
        return res
    for local_dir in root.glob("*/local"):
        for p in sorted(local_dir.glob("*.json")):
            if p.stem not in ("summary", "model_meta"):
                res[p.stem] = _main_score(p)
        sp = local_dir / "summary.json"
        if sp.exists():
            for k, v in json.loads(sp.read_text()).get("results", {}).items():
                res[k] = v
    return res

# fixed column order: by TYPE_ORDER, alphabetical within type, core-first then extra
expected = FMA_SUITE_SOTA + EXTRA_SOTA
by_type = defaultdict(list)
for t in expected:
    by_type[TASK_TYPE_SOTA.get(t, "Other")].append(t)
raw_ordered = []
for ttype in TYPE_ORDER_SOTA:
    raw_ordered += sorted(by_type.get(ttype, []))
ordered = [t for t in raw_ordered if t in _CORE_SET] + [t for t in raw_ordered if t not in _CORE_SET]

cols = pd.MultiIndex.from_tuples(
    [(TASK_TYPE_SOTA.get(t, "Other"), t) for t in ordered] + [("", "AVG")], names=["Type", "Task"])
rows, idx = [], []
for lbl, root in MAEB_ROOTS_SOTA.items():
    res = _load_maeb(root)
    vals = [res.get(t, float("nan")) for t in ordered]
    fin = [res.get(t) for t in FMA_SUITE_SOTA if res.get(t) is not None]
    vals.append(sum(fin) / len(FMA_SUITE_SOTA) if fin else float("nan"))
    rows.append(vals); idx.append(lbl)
df_maeb_sota = pd.DataFrame(rows, index=idx, columns=cols)
df_maeb_sota.index.name = "model"

def _hl_max(s):
    f = s[s.notna()]
    if f.empty:
        return [""] * len(s)
    return ["font-weight:bold; color:#4C72B0" if (pd.notna(v) and v == f.max()) else "" for v in s]

print("\n  SOTA BASELINES — MAEB (FMA suite + extra, 64-d, higher-is-better)\n")
display(df_maeb_sota.style.apply(_hl_max, axis=0).format("{:.4f}", na_rep="—"))


  SOTA BASELINES — reconstruction (FMA-large test)



,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud,n_files
model,,,,,,,,,
codicodec,-2.1012,1.7822,1.3786,0.9071,0.9337,0.060,0.2831,0.0258,11244
music2latent,-3.8818,1.2042,1.5312,0.8617,0.9060,0.153,0.3955,0.0447,11244
same,10.3494,10.8412,1.7444,0.8965,0.9267,0.041,0.2117,0.0376,11244
sao-vae,4.3924,5.6401,1.3421,0.8919,0.9113,0.069,0.3191,0.0442,11244



  SOTA BASELINES — MAEB (FMA suite + extra, 64-d, higher-is-better)



In [14]:
# ── MOISESDB RECONSTRUCTION (Mixtures & Stems) ───────────────────────────────
for split in ["mixtures", "stems"]:
    roots_to_scan = [
        Path(f"../runs/eval_sota_moisesdb_{split}"),
        Path(f"../runs/eval_sao_moisesdb_{split}"),
        Path(f"../runs/eval_swin_large2_moisesdb_{split}")

    ]
    rows = []
    
    for root in roots_to_scan:
        if not root.exists():
            continue
        for model_dir in root.iterdir():
            if model_dir.is_dir() and (model_dir / "metrics" / "_done").exists():
                name = f"{model_dir.name} ({'SOTA' if 'sota' in root.name else 'Ours'})"
                d = _load_recon(model_dir)
                row = {"model": name}
                for k in PER_FILE_SOTA:
                    row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
                for k in SCALAR_SOTA:
                    row[k] = d.get(k, float("nan"))
                row["n_files"] = int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
                rows.append(row)

    if rows:
        summary_moises = pd.DataFrame(rows).set_index("model")
        print(f"\n  MOISESDB {split.upper()} — reconstruction\n")
        display(summary_moises.style
                .apply(_hl_sota, subset=PER_FILE_SOTA + SCALAR_SOTA)
                .format({**{k: "{:.4f}" for k in PER_FILE_SOTA + SCALAR_SOTA},
                         "cdpam": "{:.3f}", "n_files": "{:.0f}"}, na_rep="—"))
    else:
        print(f"\n  MOISESDB {split.upper()} — No evaluations found yet.")



  MOISESDB MIXTURES — reconstruction



,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud,n_files
model,,,,,,,,,
codicodec (SOTA),1.0066,3.2791,0.9665,0.9320,0.9493,0.057,0.3895,0.0651,240
music2latent (SOTA),-0.3387,2.6085,1.1288,0.8882,0.9255,0.108,0.5525,0.0751,240
same (SOTA),11.5787,11.9150,1.0334,0.9251,0.9495,0.036,0.3226,0.0399,240
sao-vae (SOTA),6.1284,6.8919,0.9581,0.9311,0.9438,0.055,0.3880,0.0470,240
sa2Disc-step=500000 (Ours),4.4377,5.0954,1.0820,0.8411,0.8626,0.075,0.7770,0.1510,240
LARGE2_mrmel05_e99 (Ours),3.9064,5.0705,1.0830,0.9555,0.9680,0.091,0.2601,0.0212,240



  MOISESDB STEMS — reconstruction



,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud,n_files
model,,,,,,,,,
sao-vae (SOTA),-3.9193,5.6065,2.3689,0.8917,0.9058,0.037,9.8322,0.0874,1211
music2latent (SOTA),-9.2297,1.4462,3.6325,0.8294,0.7752,0.069,6.0655,0.3011,1211
codicodec (SOTA),-6.9330,2.7523,3.1156,0.8799,0.8709,0.040,5.4781,0.1590,1213
same (SOTA),3.1090,12.1724,2.9622,0.8849,0.8852,0.037,9.7394,0.0901,1213
sa2Disc-step=500000 (Ours),-6.6327,3.4133,2.8729,0.8051,0.8101,0.062,7.0164,0.2181,1213
LARGE2_mrmel05_e99 (Ours),-8.2891,3.6032,2.9573,0.9189,0.9196,0.070,6.9112,0.0545,1213


In [1]:
# ════════════════════════════════════════════════════════════════════════════
# ── MOISESDB MAEB Results (All models) ──────────────────────────────────────
# ════════════════════════════════════════════════════════════════════════════
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

MOISESDB_ROOTS = {
    "sao (ours)": Path("../runs/maeb_sao"),
    "swin (ours)": Path("../runs/maeb_swin"),
    "codicodec (sota)": Path("../runs/maeb_codicodec"),
    "music2latent (sota)": Path("../runs/maeb_music2latent"),
    "sao-vae (sota)": Path("../runs/maeb_sao-vae"),
    "same (sota)": Path("../runs/maeb_same")
}

MOISESDB_TASKS = [
    "MoisesDBGenreClassification", "MoisesDBGenreClustering", "MoisesDBArtistClustering",
    "MoisesDBArtistA2ARetrieval", "MoisesDBGenreAudioReranking", "MoisesDBArtistPairClassification",
    "MoisesDBInstrumentClassification"
]

def _load_moisesdb_maeb(root: Path):
    if not root.exists(): return {}
    # Trova l'eval più recente (nel caso di cvae ne abbiamo tante)
    dirs = [d / "local" for d in root.iterdir() if d.is_dir() and (d / "local").is_dir()]
    if not dirs: return {}
    latest = sorted(dirs, key=lambda d: d.stat().st_mtime)[-1]
    
    res = {}
    for p in latest.glob("MoisesDB*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
                scores = d.get("scores", {})
                val = None
                for split_data in scores.values():
                    if isinstance(split_data, list) and len(split_data) > 0 and "main_score" in split_data[0]:
                        val = split_data[0]["main_score"]
                        break
                res[p.stem] = val if val is not None else float('nan')
        except: pass
    return res

rows, idx = [], []
for lbl, root in MOISESDB_ROOTS.items():
    res = _load_moisesdb_maeb(root)
    vals = [res.get(t, float("nan")) for t in MOISESDB_TASKS]
    
    fin = [v for v in vals if pd.notna(v)]
    avg = sum(fin) / len(MOISESDB_TASKS) if fin else float("nan")
    
    rows.append(vals + [avg])
    idx.append(lbl)

cols = MOISESDB_TASKS + ["AVG_MoisesDB"]
df_moisesdb = pd.DataFrame(rows, index=idx, columns=cols)
df_moisesdb.index.name = "model"

def _hl_max(s):
    f = s[s.notna()]
    if f.empty: return [''] * len(s)
    m = f.max()
    return ['font-weight: bold; color: #4C72B0' if v == m else '' for v in s]

print("\n  MOISESDB MAEB — FMA Suite transferred to MoisesDB chunks_30s (higher-is-better)\n")
display(df_moisesdb.style.apply(_hl_max, axis=0).format("{:.4f}", na_rep="—"))



  MOISESDB MAEB — FMA Suite transferred to MoisesDB chunks_30s (higher-is-better)



,MoisesDBGenreClassification,MoisesDBGenreClustering,MoisesDBArtistClustering,MoisesDBArtistA2ARetrieval,MoisesDBGenreAudioReranking,MoisesDBArtistPairClassification,MoisesDBInstrumentClassification,AVG_MoisesDB
model,,,,,,,,
sao (ours),0.3778,0.1167,0.4667,0.4960,0.5608,0.6776,0.5024,0.4568
swin (ours),—,—,—,—,—,—,—,—
codicodec (sota),0.4000,0.1120,0.4671,0.4910,0.5570,0.6476,0.5160,0.4558
music2latent (sota),0.4267,0.1316,0.4717,0.5181,0.5732,0.6797,0.5550,0.4794
sao-vae (sota),0.4089,0.1553,0.4788,0.5174,0.5615,0.6791,0.4801,0.4687
same (sota),0.4533,0.1570,0.4660,0.5241,0.5878,0.6974,0.5630,0.4927


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# CHUNKS_MIX_ORIGINAL — reconstruction (10s chunks, 1998 files)
# SELF-CONTAINED: run this cell alone (no dependency on cells above).
#
# Modelli:
#   SOTA  → runs/eval_sota_10s/{codicodec,music2latent,sao-vae,same}/
#   SAO   → runs/eval_sao_10s/sa2Disc-step=500000/
#   LARGE2→ runs/eval_swin_10s/LARGE2_mrmel05_e99/
#
# Metriche CLAP/MERT calcolate con ref pre-computate dal collega;
# MERT usa layer=4 per tutti i modelli (confronto diretto).
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import pandas as pd
import numpy as np

def _load_recon_10s(p: Path) -> dict:
    """Load all metric CSVs from a model's metrics/ directory."""
    m, d = p / "metrics", {}
    for fname, cols in [("spectral.csv", ["si_sdr", "sdr", "stft_loss"]),
                         ("cdpam.csv", ["cdpam"])]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            for c in cols:
                if c in df.columns:
                    v = df[c].dropna().values.astype(float)
                    d[c] = v[np.isfinite(v)]
    for key, fname in [("clap_music", "clap_music.csv"), ("clap_audio", "clap_audio.csv")]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            if "cosine" in df.columns:
                v = df["cosine"].dropna().values.astype(float)
                d[key] = v[np.isfinite(v)]
    for fname, key in [("fad_mert.csv", "fad_mert"),
                        ("fad_gudgud.csv", "fad_gudgud")]:
        if (m / fname).exists():
            try:
                d[key] = float(pd.read_csv(m / fname).iloc[0]["score"])
            except Exception:
                pass
    return d

# ── model registry ────────────────────────────────────────────────────────────
# (display_name, root_dir, subdir, type_tag)
_BASE = Path("../runs")
MODELS_10S = [
    ("codicodec",           _BASE / "eval_sota_10s",  "codicodec",           "SOTA"),
    ("music2latent",        _BASE / "eval_sota_10s",  "music2latent",        "SOTA"),
    ("sao-vae (SAO open)",  _BASE / "eval_sota_10s",  "sao-vae",             "SOTA"),
    ("same",                _BASE / "eval_sota_10s",  "same",                "SOTA"),
    ("sa2Disc-500k (ours)", _BASE / "eval_sao_10s",   "sa2Disc-step=500000", "Ours"),
    ("LARGE2_e99 (ours)",   _BASE / "eval_swin_10s",  "LARGE2_mrmel05_e99",  "Ours"),
    ("FT e99 (ours)",       _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e99",  "Ours"),
    ("FT e199 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e199", "Ours"),
    ("FT e219 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e219", "Ours"),
    ("FT e239 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e239", "Ours"),
    ("FT e259 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e259", "Ours"),
    ("FT e279 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e279", "Ours"),
    ("FT e299 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e299", "Ours"),
    ("FT e319 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e319", "Ours"),
    ("FT e339 (ours)",      _BASE / "eval_swin_10s",  "FT_LARGE2_mr05_e339", "Ours"),
    ("COOLDOWN e9 (ours)",  _BASE / "eval_swin_10s",  "COOLDOWN_LARGE2_mr05_e9", "Ours"),
    ("COOLDOWN e19 (ours)",  _BASE / "eval_swin_10s",  "COOLDOWN_LARGE2_mr05_e19", "Ours"),
    ("COOLDOWN e29 (ours)",  _BASE / "eval_swin_10s",  "COOLDOWN_LARGE2_mr05_e29", "Ours"),
    ("COOLDOWN e39 (ours)",  _BASE / "eval_swin_10s",  "COOLDOWN_LARGE2_mr05_e39", "Ours"),
    ("COOLDOWN e49 (ours)",  _BASE / "eval_swin_10s",  "COOLDOWN_LARGE2_mr05_e49", "Ours"),


]

PER_FILE_10S      = ["si_sdr", "sdr", "stft_loss", "clap_music", "clap_audio", "cdpam"]
SCALAR_10S        = ["fad_mert", "fad_gudgud"]
HIGHER_BETTER_10S = {"si_sdr", "sdr", "clap_music", "clap_audio"}
ALL_METRICS_10S   = PER_FILE_10S + SCALAR_10S

def _hl_10s(s):
    finite = s[s.notna()]
    if finite.empty:
        return [""] * len(s)
    # separate SOTA vs Ours for highlight? No — highlight global best across all models
    best = finite.max() if s.name in HIGHER_BETTER_10S else finite.min()
    return ["font-weight:bold; color:#E84393" if (pd.notna(v) and v == best) else ""
            for v in s]

rows = []
for display_name, root, subdir, tag in MODELS_10S:
    model_dir = root / subdir
    d         = _load_recon_10s(model_dir) if model_dir.exists() else {}
    row       = {"model": display_name, "type": tag,
                 "n_files": int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))}
    for k in PER_FILE_10S:
        row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
    for k in SCALAR_10S:
        row[k] = d.get(k, float("nan"))
    rows.append(row)

df_10s = pd.DataFrame(rows).set_index("model")

print("\n  CHUNKS_MIX_ORIGINAL (10s) — reconstruction — tutti i modelli\n")
print("  Ref stats pre-calcolate dal collega | MERT layer=4 per tutti\n")
display(df_10s.style
        .apply(_hl_10s, subset=ALL_METRICS_10S)
        .format({**{k: "{:.4f}" for k in PER_FILE_10S + SCALAR_10S},
                 "cdpam": "{:.4f}", "n_files": "{:.0f}"}, na_rep="—"))



  CHUNKS_MIX_ORIGINAL (10s) — reconstruction — tutti i modelli

  Ref stats pre-calcolate dal collega | MERT layer=4 per tutti



,type,n_files,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud
model,,,,,,,,,,
codicodec,SOTA,1998,-1.3847,1.8377,0.8478,0.9304,0.9233,0.0771,0.3915,0.1285
music2latent,SOTA,1998,-2.8364,1.2246,0.9431,0.9299,0.9444,0.1460,0.6192,0.3180
sao-vae (SAO open),SOTA,1998,3.7905,5.0219,0.9558,0.9333,0.9437,0.0778,0.5101,0.1181
same,SOTA,1998,8.9521,9.4925,0.8163,0.9537,0.9662,0.0554,0.2652,0.0330
sa2Disc-500k (ours),Ours,1998,2.4626,3.4561,0.8797,0.8140,0.8189,0.0934,1.5554,0.3693
LARGE2_e99 (ours),Ours,1998,2.6009,4.1112,0.9500,0.9614,0.9712,0.0946,0.3564,0.0225
FT e99 (ours),Ours,1998,2.6415,4.0682,0.8766,0.9678,0.9753,0.0970,0.3349,0.0163
FT e199 (ours),Ours,1998,2.6132,4.0644,0.8799,0.9683,0.9759,0.1042,0.2991,0.0159
FT e219 (ours),Ours,1998,2.6245,4.0697,0.8747,0.9681,0.9759,0.1022,0.3003,0.0159


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# FT_LARGE2_mr05_decFT_postnet — INTERNAL checkpoint selection (recon only)
# SELF-CONTAINED: run this cell alone. Auto-discovers every epoch evaluated by
# evaluation/eval_ft_internal.slurm under runs/eval_ft_internal/FT_LARGE2_mr05_e<EPOCH>/.
#
# Computed on a FIXED subset of the FMA-large test split (--max-files, SAME files
# for every epoch) → ranks the finetune's OWN checkpoints consistently. NOT
# comparable to the full-test-set tables above. Pick the best (composite ↓ or by
# the metric you care about), then re-eval that checkpoint on the FULL test set.
# ════════════════════════════════════════════════════════════════════════════
import re
from pathlib import Path
import pandas as pd
import numpy as np

FT_INTERNAL_ROOT = Path("../runs/eval_ft_internal")

def _load_recon_ft(p: Path) -> dict:
    m, d = p / "metrics", {}
    for fname, cols in [("spectral.csv", ["si_sdr", "sdr", "stft_loss"]),
                        ("cdpam.csv", ["cdpam"])]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            for c in cols:
                if c in df.columns:
                    v = df[c].dropna().values.astype(float)
                    d[c] = v[np.isfinite(v)]
    for key, fname in [("clap_music", "clap_music.csv"), ("clap_audio", "clap_audio.csv")]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            if "cosine" in df.columns:
                v = df["cosine"].dropna().values.astype(float)
                d[key] = v[np.isfinite(v)]
    for fname, key in [("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud")]:
        if (m / fname).exists():
            try:
                d[key] = float(pd.read_csv(m / fname).iloc[0]["score"])
            except Exception:
                pass
    return d

PER_FILE_FT      = ["si_sdr", "sdr", "stft_loss", "clap_music", "clap_audio", "cdpam"]
SCALAR_FT        = ["fad_mert", "fad_gudgud"]
HIGHER_BETTER_FT = {"si_sdr", "sdr", "clap_music", "clap_audio"}
ALL_METRICS_FT   = PER_FILE_FT + SCALAR_FT

def _epoch_of(name: str) -> int:
    # FT checkpoints → their epoch; any other dir (LARGE2 start baseline) → 0 (sorts first)
    m = re.match(r"FT_LARGE2_mr05_e(\d+)$", name)
    return int(m.group(1)) if m else 0

def _hl_ft(s):
    finite = s[s.notna()]
    if finite.empty:
        return [""] * len(s)
    best = finite.max() if s.name in HIGHER_BETTER_FT else finite.min()  # composite ↓
    return ["font-weight:bold; color:#E84393" if (pd.notna(v) and v == best) else "" for v in s]

if FT_INTERNAL_ROOT.exists():
    dirs = sorted([p for p in FT_INTERNAL_ROOT.iterdir() if p.is_dir()],
                  key=lambda p: _epoch_of(p.name))
    rows = []
    for p in dirs:
        d = _load_recon_ft(p)
        row = {"checkpoint": p.name, "epoch": _epoch_of(p.name),
               "n_files": int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))}
        for k in PER_FILE_FT:
            row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
        for k in SCALAR_FT:
            row[k] = d.get(k, float("nan"))
        rows.append(row)

    df_ft = pd.DataFrame(rows).set_index("checkpoint")
    df_ft["composite"] = df_ft["fad_mert"] + df_ft["fad_gudgud"] + df_ft["cdpam"]  # ↓ better

    n_ft = int(df_ft["n_files"].max()) if len(df_ft) else 0
    print(f"\n  FT_LARGE2_mr05_decFT_postnet — INTERNAL recon ({n_ft} files/ckpt, subset → ranking only)\n")
    df_ft = df_ft.sort_values("epoch")
    display(df_ft.style
            .apply(_hl_ft, subset=ALL_METRICS_FT + ["composite"])
            .format({**{k: "{:.4f}" for k in ALL_METRICS_FT},
                     "cdpam": "{:.4f}", "composite": "{:.4f}",
                     "n_files": "{:.0f}", "epoch": "{:.0f}"}, na_rep="—"))
else:
    print(f"Not found: {FT_INTERNAL_ROOT} — run evaluation/eval_ft_internal.slurm first")



  FT_LARGE2_mr05_decFT_postnet — INTERNAL recon (2498 files/ckpt, subset → ranking only)



,epoch,n_files,si_sdr,sdr,stft_loss,clap_music,clap_audio,cdpam,fad_mert,fad_gudgud,composite
checkpoint,,,,,,,,,,,
LARGE2_mrmel05_e99,0,2498,1.5777,3.8719,1.0498,0.9330,0.9361,0.0876,0.2544,0.0301,0.3721
COOLDOWN_LARGE2_mr05_e49,0,2498,1.7709,3.8442,1.0280,0.9423,0.9433,0.0914,0.2190,0.0246,0.3350
FT_LARGE2_mr05_e19,19,2498,1.6659,3.8803,1.0646,0.9334,0.9369,0.0893,0.2695,0.0297,0.3884
FT_LARGE2_mr05_e39,39,2498,1.6494,3.8563,1.0519,0.9368,0.9388,0.0898,0.2572,0.0281,0.3752
FT_LARGE2_mr05_e59,59,2498,1.6441,3.8511,1.0507,0.9382,0.9396,0.0898,0.2491,0.0275,0.3664
FT_LARGE2_mr05_e79,79,2498,1.6443,3.8487,1.0432,0.9391,0.9400,0.0900,0.2473,0.0268,0.3640
FT_LARGE2_mr05_e99,99,2498,1.6801,3.8505,1.0310,0.9397,0.9406,0.0903,0.2408,0.0267,0.3577
FT_LARGE2_mr05_e119,119,2498,1.6864,3.8563,1.0278,0.9400,0.9410,0.0900,0.2355,0.0260,0.3514
FT_LARGE2_mr05_e139,139,2498,1.6816,3.8590,1.0247,0.9406,0.9411,0.0897,0.2349,0.0263,0.3509


In [10]:
# ════════════════════════════════════════════════════════════════════════════
# MusicCaps & SDD — reconstruction (10s, frozen SOTA-comparison sets)
# SELF-CONTAINED: run this cell alone.
#
#   SOTA  → runs/eval_sota_10s/<tag>/{codicodec,music2latent,same,sao-vae}/
#   SWIN  → runs/eval_swin_10s/<tag>/epoch_239/
#   tag = balanced_44k_stereo (MusicCaps, 964) | sdd_44k_stereo_10s (SDD, 8364)
#
# Same pipeline as the moisesdb 10s cell, but refs = clean-original embeddings +
# OUR Fréchet stats (stats_ours/) built by build_refs_10s.py. MERT layer=4 for
# all models → directly comparable. (CSVs appear as jobs finish; missing → "—".)
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import pandas as pd
import numpy as np

def _load_recon_10s(p: Path) -> dict:
    m, d = p / "metrics", {}
    for fname, cols in [("spectral.csv", ["si_sdr", "sdr", "stft_loss", "mel_loss"]),
                        ("cdpam.csv", ["cdpam"])]:

        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            for c in cols:
                if c in df.columns:
                    v = df[c].dropna().values.astype(float); d[c] = v[np.isfinite(v)]
    for key, fname in [("clap_music", "clap_music.csv"), ("clap_audio", "clap_audio.csv")]:
        if (m / fname).exists():
            df = pd.read_csv(m / fname)
            if "cosine" in df.columns:
                v = df["cosine"].dropna().values.astype(float); d[key] = v[np.isfinite(v)]
    for fname, key in [("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud"), ("fad_pann.csv", "fad_pann")]:
        if (m / fname).exists():
            try: d[key] = float(pd.read_csv(m / fname).iloc[0]["score"])
            except Exception: pass
    return d

_BASE = Path("../runs")
def _registry(tag):
    # (display_name, root, subdir, type)
    return [
        ("codicodec",           _BASE / "eval_sota_10s" / tag, "codicodec",    "SOTA"),
        ("music2latent",        _BASE / "eval_sota_10s" / tag, "music2latent", "SOTA"),
        ("same",                _BASE / "eval_sota_10s" / tag, "same",         "SOTA"),
        ("sao-vae (SAO open)",  _BASE / "eval_sota_10s" / tag, "sao-vae",      "SOTA"),
        ("SAGE e992 tri2", _BASE / "eval_swin_10s" / tag, "SAGE_e992_tri2", "Ours"),
        ("same-s",              _BASE / "eval_sota_10s" / tag, "same-s",       "SOTA"),

    ]

PER_FILE_10S = ["sdr", "stft_loss", "mel_loss", "cdpam", "clap_music", "clap_audio"]
SCALAR_10S        = ["fad_mert", "fad_gudgud", "fad_pann"]
HIGHER_BETTER_10S = {"si_sdr", "sdr", "clap_music", "clap_audio"}
ALL_METRICS_10S   = PER_FILE_10S + SCALAR_10S

def _hl_10s(s):
    finite = s[s.notna()]
    if finite.empty: return [""] * len(s)
    best = finite.max() if s.name in HIGHER_BETTER_10S else finite.min()
    return ["font-weight:bold; color:#E84393" if (pd.notna(v) and v == best) else "" for v in s]

DATASETS = [("MusicCaps (balanced, 964)",   "balanced_44k_stereo"),
            ("Song Describer (full, 8364)", "sdd_44k_stereo_10s")]

for title, tag in DATASETS:
    rows = []
    for display_name, root, subdir, typ in _registry(tag):
        mdir = root / subdir
        d    = _load_recon_10s(mdir) if mdir.exists() else {}
        row  = {"model": display_name, "type": typ, "n_files": int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))}
        for k in PER_FILE_10S: row[k] = float(np.mean(d[k])) if k in d and len(d[k]) else float("nan")
        for k in SCALAR_10S:   row[k] = d.get(k, float("nan"))
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model")
    print(f"\n  {title} — reconstruction (10s) — clean-original refs + stats_ours | MERT l4\n")
    display(df.style
            .apply(_hl_10s, subset=ALL_METRICS_10S)
            .format({**{k: "{:.4f}" for k in PER_FILE_10S + SCALAR_10S},
                     "cdpam": "{:.4f}", "n_files": "{:.0f}"}, na_rep="—"))



  MusicCaps (balanced, 964) — reconstruction (10s) — clean-original refs + stats_ours | MERT l4



,type,n_files,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann
model,,,,,,,,,,,
codicodec,SOTA,964,1.8302,1.2385,0.7074,0.0669,0.9182,0.9403,0.2837,0.0557,5.6350
music2latent,SOTA,964,1.1176,1.4063,0.8136,0.1622,0.8871,0.9283,0.5816,0.2867,5.7132
same,SOTA,964,10.5608,1.7414,0.8170,0.0487,0.8695,0.9139,0.5075,0.0920,3.5293
sao-vae (SAO open),SOTA,964,5.4699,1.2584,0.6641,0.0763,0.8665,0.9093,0.5287,0.1252,5.1714
SAGE e992 tri2,Ours,964,5.0916,1.2290,0.5829,0.0598,0.9302,0.9574,0.3057,0.0290,3.1690
same-s,SOTA,964,8.5562,1.6556,0.8383,0.0809,0.8276,0.8733,0.9569,0.1338,6.4840



  Song Describer (full, 8364) — reconstruction (10s) — clean-original refs + stats_ours | MERT l4



,type,n_files,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann
model,,,,,,,,,,,
codicodec,SOTA,8364,2.4178,1.1784,0.5931,0.0594,0.9221,0.9195,0.2898,0.0848,8.3974
music2latent,SOTA,8364,2.2394,1.2902,0.6618,0.1343,0.9154,0.9350,0.4082,0.2756,2.8465
same,SOTA,8364,12.1675,1.4509,0.5964,0.0376,0.9398,0.9547,0.1860,0.0373,1.4967
sao-vae (SAO open),SOTA,8364,6.8597,1.2128,0.5641,0.0648,0.9237,0.9420,0.3494,0.1264,2.4168
SAGE e992 tri2,Ours,8364,6.3123,0.8783,0.4076,0.0521,0.9694,0.9758,0.0979,0.0112,0.8882
same-s,SOTA,8364,9.9738,1.3985,0.6264,0.0669,0.8962,0.9150,0.6484,0.0869,4.7317


In [11]:
# =====================================================================
# FINAL EVAL — paper tables (SELF-CONTAINED). Reads ONLY runs/final_eval/.
# 5 tables: (1) FMA recon, (2) MoisesDB-10s recon, (3) FMA MAEB, (4) MoisesDB MAEB, (5) upstream MAEB music
# Recon tables also report n_files (full-set correctness check) + pure inference
# timing (infer_ms/file median, RTF = infer_sec/audio_sec). MAEB = task main_score.
# =====================================================================
import csv as _csv, json
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown

FE = Path("../runs/final_eval")
MODELS = ["sao-vae", "codicodec", "music2latent", "same",
        #    "SAGE_single_dataset", "SAGE_final_1499",
          "SAGE_e992_tri2", "same-s"]

_RECON_DIR = {"SAGE_single_dataset": "SAGE_e299",
              "SAGE_e1137_tri2": "FT_A_mrstft_sd_e1137_tri2"}
def _recon_subdir(m):  return _RECON_DIR.get(m, m)
def _maeb_root(m):     return FE / "maeb" / ("maeb_sage" if m == "SAGE_single_dataset" else f"maeb_{m}")
def _maeb_label(m):    return "same*" if m == "same" else m               # 256-d caveat marker

# ---------- reconstruction loaders (mirror load_checkpoint, cells 0/9) ----------
def _col(path, col):
    out = []
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: out.append(float(r[col]))
            except (KeyError, ValueError, TypeError): pass
    return np.asarray(out, dtype=float)

def _scalar(path, col="score"):
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: return float(r[col])
            except (KeyError, ValueError, TypeError): return float("nan")
    return float("nan")

def load_recon(metrics: Path) -> dict:
    d = {}
    sp = metrics / "spectral.csv"
    if sp.exists():
        for k in ("si_sdr", "sdr", "stft_loss", "mel_loss"): d[k] = _col(sp, k)
    if (metrics / "cdpam.csv").exists(): d["cdpam"] = _col(metrics / "cdpam.csv", "cdpam")
    for fn, k in (("clap_music.csv", "clap_music"), ("clap_audio.csv", "clap_audio")):
        if (metrics / fn).exists(): d[k] = _col(metrics / fn, "cosine")
    if (metrics / "clap_score.csv").exists():                              # legacy combined
        for k in ("clap_music", "clap_audio"):
            d.setdefault(k, _col(metrics / "clap_score.csv", k))
    for fn, k in (("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud"), ("fad_pann.csv", "fad_pann")):
        if (metrics / fn).exists(): d[k] = _scalar(metrics / fn, "score")
    if (metrics / "timing.csv").exists():
        d["_infer"] = _col(metrics / "timing.csv", "infer_sec")
        d["_audio"] = _col(metrics / "timing.csv", "audio_sec")
    return d

PERFILE = ["si_sdr", "sdr", "stft_loss", "mel_loss", "cdpam", "clap_music", "clap_audio"]
SCALAR  = ["fad_mert", "fad_gudgud", "fad_pann"]
HIGHER  = {"si_sdr", "sdr", "clap_music", "clap_audio"}        # else lower-is-better

def recon_table(root: Path, title: str):
    rows = []
    for m in MODELS:
        met = root / _recon_subdir(m) / "metrics"
        row = {"model": m}
        if met.exists():
            d = load_recon(met)
            for k in PERFILE: 
                arr = np.array(d.get(k, []))
                arr = arr[np.isfinite(arr)] # scarta i brani muti (-inf) di FMA!
                row[k] = float(np.mean(arr)) if len(arr) > 0 else float("nan")

            for k in SCALAR:  row[k] = d.get(k, float("nan"))
            row["n_files"] = int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
            it = d.get("_infer", np.array([]))
            if len(it):
                row["infer_ms/file"] = float(np.median(it)) * 1e3
                at = d.get("_audio")
                row["RTF"] = float(np.median(it / np.clip(at, 1e-9, None))) if at is not None and len(at) == len(it) else float("nan")
            else:
                row["infer_ms/file"] = row["RTF"] = float("nan")
        rows.append(row)
    disp = PERFILE + SCALAR + ["n_files", "infer_ms/file", "RTF"]
    df = pd.DataFrame(rows).set_index("model").reindex(columns=disp)
    lower = (set(PERFILE + SCALAR) - HIGHER) | {"infer_ms/file", "RTF"}
    def hl(s):
        if s.name == "n_files": return [""] * len(s)
        fin = s[s.notna()]
        if fin.empty: return [""] * len(s)
        best = fin.min() if s.name in lower else fin.max()
        return ["font-weight:bold;color:#00cc66" if (pd.notna(v) and v == best) else "" for v in s]
    fmt = {**{c: "{:.4f}" for c in PERFILE + SCALAR}, "cdpam": "{:.4f}",
           "n_files": "{:.0f}", "infer_ms/file": "{:.1f}", "RTF": "{:.3f}"}
    display(Markdown(f"### {title}"))
    display(df.style.apply(hl).format(fmt, na_rep="—"))
    return df

# ---------- MAEB loaders (mirror cells 3/6/8) ----------
def _main_score(path: Path):
    d = json.loads(path.read_text())
    for sv in d.get("scores", {}).values():
        if isinstance(sv, list):
            for s in sv:
                if isinstance(s, dict) and "main_score" in s: return float(s["main_score"])
        elif isinstance(sv, dict) and "main_score" in sv:
            return float(sv["main_score"])
    return None

def load_maeb(root: Path) -> dict:
    res = {}
    for local in root.glob("*/local"):
        for jp in local.glob("*.json"):
            if jp.stem in ("summary", "model_meta"): continue
            sc = _main_score(jp)
            if sc is not None: res[jp.stem] = sc
    return res

CORE_FMA = ["FMAGenreClassification", "FMAGenreClustering", "FMAArtistClustering",
            "FMAArtistA2ARetrieval", "FMAGenreAudioReranking", "FMAArtistPairClassification"]
MAEB_MUSIC = ["GTZANGenre", "GTZANGenreClustering", "MusicGenreClustering",
              "GTZANAudioReranking", "NSynth", "JamAltArtistA2ARetrieval"]
MOIS     = ["MoisesDBGenreClassification", "MoisesDBGenreClustering", "MoisesDBArtistClustering",
            "MoisesDBArtistA2ARetrieval", "MoisesDBGenreAudioReranking",
            "MoisesDBArtistPairClassification", "MoisesDBInstrumentClassification"]

def maeb_table(tasks, core, title, note=None):
    data = {m: load_maeb(_maeb_root(m)) for m in MODELS}
    rows = []
    for m in MODELS:
        r = [data[m].get(t) for t in tasks]
        fin = [v for v in (data[m].get(t) for t in core) if v is not None]
        r.append(sum(fin) / len(core) if core else float("nan"))
        rows.append(r)
    df = pd.DataFrame(rows, index=[_maeb_label(m) for m in MODELS], columns=tasks + ["AVG"])
    def hl(s):
        fin = s[s.notna()]
        if fin.empty: return [""] * len(s)
        best = fin.max()
        return ["font-weight:bold;color:#4C72B0" if (pd.notna(v) and v == best) else "" for v in s]
    display(Markdown(f"### {title}"))
    display(df.style.apply(hl, axis=0).format("{:.4f}", na_rep="—"))
    if note: display(Markdown(note))
    return df

# ---------- render ----------
display(Markdown("# FINAL EVAL — paper tables  \n_Self-contained; reads `runs/final_eval/` only._"))
recon_fma  = recon_table(FE / "recon_fma",            "1) FMA — reconstruction (full test split)")
recon_mois = recon_table(FE / "recon_moisesdb_10s",  "2) MoisesDB-10s — reconstruction (full 1998)")
recon_stems = recon_table(FE / "recon_moisesdb_stems_10s", "2b) MoisesDB-stems 10s — reconstruction (1546)")
_NOTE = ("*`same*` = native 256-d (~×32 compression, **not** width-matched vs the 64-d models) "
         "→ MAEB scores are upper-biased, especially classification.*")
maeb_fma   = maeb_table(CORE_FMA, CORE_FMA, "3) FMA — MAEB probing (6 tasks, AVG over all)", _NOTE)
maeb_mois  = maeb_table(MOIS, MOIS,          "4) MoisesDB 30s — MAEB probing (7 tasks, AVG over all)", _NOTE)
maeb_music = maeb_table(MAEB_MUSIC, MAEB_MUSIC, "5) Upstream MAEB music — original datasets (6 tasks, AVG over all)", _NOTE)


# FINAL EVAL — paper tables  
_Self-contained; reads `runs/final_eval/` only._

### 1) FMA — reconstruction (full test split)

,si_sdr,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann,n_files,infer_ms/file,RTF
model,,,,,,,,,,,,,
sao-vae,4.3924,5.6401,1.3421,0.6382,0.0694,0.8919,0.9113,0.3191,0.0437,2.8905,11252,278.4,0.009
codicodec,-2.1050,1.7804,1.3787,0.6834,0.0598,0.9072,0.9340,0.2829,0.0256,3.2434,11252,1213.1,0.040
music2latent,-3.8796,1.2050,1.5311,0.7811,0.1534,0.8618,0.9061,0.3955,0.0452,4.5990,11252,1654.9,0.055
same,10.4283,10.9123,1.7461,0.7348,0.0405,0.8968,0.9270,0.2102,0.0376,3.4701,10891,2982.4,0.099
SAGE_e992_tri2,4.1447,5.2841,0.9291,0.4418,0.0530,0.9559,0.9602,0.1129,0.0161,1.4218,11252,151.3,0.005
same-s,8.1352,8.8007,1.6758,0.7605,0.0725,0.8396,0.8701,0.6488,0.0817,7.1690,11252,146.8,0.005


### 2) MoisesDB-10s — reconstruction (full 1998)

,si_sdr,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann,n_files,infer_ms/file,RTF
model,,,,,,,,,,,,,
sao-vae,3.7905,5.0219,0.9558,0.4782,0.0778,0.9333,0.9437,0.5101,0.1181,2.4765,1998,97.8,0.010
codicodec,-1.3816,1.8367,0.8478,0.4940,0.0774,0.9303,0.9232,0.3906,0.1299,11.5313,1998,444.6,0.044
music2latent,-2.8454,1.2213,0.9431,0.5581,0.1463,0.9302,0.9445,0.6185,0.3184,3.3620,1998,519.3,0.052
same,8.9522,9.4925,0.8163,0.4005,0.0553,0.9537,0.9662,0.2652,0.0331,1.2772,1998,211.7,0.021
SAGE_e992_tri2,3.8515,4.7873,0.8565,0.3828,0.0609,0.9656,0.9714,0.2053,0.0264,1.4383,1998,56.0,0.006
same-s,7.3060,7.9687,0.8394,0.4457,0.1003,0.9141,0.9299,1.0244,0.0936,4.2659,1998,32.4,0.003


...

,MoisesDBGenreClassification,MoisesDBGenreClustering,MoisesDBArtistClustering,MoisesDBArtistA2ARetrieval,MoisesDBGenreAudioReranking,MoisesDBArtistPairClassification,MoisesDBInstrumentClassification,AVG
sao-vae,0.4089,0.1553,0.4788,0.5174,0.5615,0.6791,0.4801,0.4687
codicodec,0.4000,0.1120,0.4671,0.4910,0.5570,0.6476,0.5160,0.4558
music2latent,0.4267,0.1316,0.4717,0.5181,0.5732,0.6797,0.5550,0.4794
same*,0.4533,0.1570,0.4660,0.5241,0.5878,0.6974,0.5630,0.4927
SAGE_e992_tri2,0.5333,0.1934,0.5324,0.5742,0.6074,0.7438,0.6244,0.5441
same-s,0.4444,0.1414,0.4746,0.5172,0.5797,0.6945,0.5598,0.4874


*`same*` = native 256-d (~×32 compression, **not** width-matched vs the 64-d models) → MAEB scores are upper-biased, especially classification.*

### 5) Upstream MAEB music — original datasets (6 tasks, AVG over all)

,GTZANGenre,GTZANGenreClustering,MusicGenreClustering,GTZANAudioReranking,NSynth,JamAltArtistA2ARetrieval,AVG
sao-vae,0.5000,0.3113,0.1390,0.5574,0.4186,0.9611,0.4813
codicodec,0.5290,0.3365,0.1818,0.5881,0.4323,0.7603,0.4713
music2latent,0.4940,0.3362,0.1439,0.5729,0.4057,0.9668,0.4866
same*,0.5310,0.2715,0.1261,0.5546,0.3960,0.9601,0.4732
SAGE_e992_tri2,0.7740,0.4705,0.3439,0.7219,0.4531,0.9672,0.6218
same-s,0.5300,0.2693,0.1253,0.5542,0.3962,0.9592,0.4724


*`same*` = native 256-d (~×32 compression, **not** width-matched vs the 64-d models) → MAEB scores are upper-biased, especially classification.*

In [4]:
# =====================================================================
# CLAP-ORACLE reference row — anti-circularity check (§4.5 defense).
# SELF-CONTAINED-ish: reuses _maeb_root()/load_maeb()/CORE_FMA/MOIS/MAEB_MUSIC
# from the FINAL EVAL cell above (run that cell first).
#
# The oracle is the FROZEN distillation teacher (music_audioset_epoch_15,
# same checkpoint as CLAPTeacher in loss_manager.py:391) probed on the exact
# same 19 MAEB tasks as SAGE. NOT a baseline — CLAP reconstructs no audio, so
# it never appears in the recon tables above. Purpose: bound how much of the
# teacher's semantic power survives inside SAGE's 64-d invertible latent.
#
# "pretrain-risk" tasks = plausibly IN-DISTRIBUTION for CLAP's own AudioSet +
# captioned-music contrastive pretraining (GTZAN*, MusicGenreClustering,
# NSynth) — SAGE never saw labels for any of them. Retention should read
# LOWER on these if the margin is really pretraining, not just teacher size.
# =====================================================================
ORACLE_SUMMARY = FE / "maeb" / "maeb_clap_oracle" / "oracle__clap-teacher" / "music_audioset_epoch_15_esc_90.14" / "summary.json"

def clap_oracle_table(sage_tag: str = "SAGE_e992_tri2"):
    if not ORACLE_SUMMARY.exists():
        display(Markdown(f"*(no CLAP-oracle run found at `{ORACLE_SUMMARY}`)*"))
        return None
    oracle = json.loads(ORACLE_SUMMARY.read_text())
    sage = load_maeb(_maeb_root(sage_tag))

    tasks = CORE_FMA + MOIS + MAEB_MUSIC   # same 19, same order as tables 3/4/5
    rows = []
    for t in tasks:
        s, o = sage.get(t), oracle["results"].get(t)
        rows.append({
            "task": t,
            "suite": ("FMA" if t in CORE_FMA else "MoisesDB" if t in MOIS else "Upstream"),
            "SAGE": s,
            "CLAP-oracle": o,
            "retention_%": 100 * s / o if (s is not None and o) else float("nan"),
        })
    df = pd.DataFrame(rows).set_index("task")

    display(Markdown("### CLAP-oracle reference row — SAGE retention vs. the frozen distillation teacher"))
    def hl(s):
        if s.name != "retention_%": return [""] * len(s)
        return ["color:#cc6600;font-weight:bold" if pd.notna(v) else "" for v in s]
    display(df.style.apply(hl).format(
        {"SAGE": "{:.4f}", "CLAP-oracle": "{:.4f}", "retention_%": "{:.1f}"}, na_rep="—"))

    for c in oracle.get("caveats", []):
        display(Markdown(f"> ⚠️ {c}"))
    return df

clap_oracle = clap_oracle_table()


### CLAP-oracle reference row — SAGE retention vs. the frozen distillation teacher

,suite,SAGE,CLAP-oracle,retention_%
task,,,,
FMAGenreClassification,FMA,0.4466,0.5294,84.4
FMAGenreClustering,FMA,0.2153,0.3847,56.0
FMAArtistClustering,FMA,0.7058,0.7193,98.1
FMAArtistA2ARetrieval,FMA,0.5828,0.5874,99.2
FMAGenreAudioReranking,FMA,0.5346,0.6262,85.4
FMAArtistPairClassification,FMA,0.8928,0.9352,95.5
MoisesDBGenreClassification,MoisesDB,0.5333,0.6978,76.4
MoisesDBGenreClustering,MoisesDB,0.1934,0.3508,55.1
MoisesDBArtistClustering,MoisesDB,0.5324,0.6120,87.0


> ⚠️ 512-d vs SAGE 64-d: NOT width-matched (cf. SAME 256-d caveat).

> ⚠️ CLAP is contrastively pretrained on AudioSet + captioned music; GTZAN/MusicGenre/NSynth are plausibly in-distribution for it and are not for SAGE. The oracle is an OPTIMISTIC upper bound.

In [3]:
# =====================================================================
# MULTICORPUS EVALUATION
# =====================================================================
from IPython.display import display, Markdown
from pathlib import Path
import numpy as np
import pandas as pd

# 1. multicorpus models
EPOCHS = [99, 199, 299, 359, 399, 499, 599, 699, 799, 899, 999, 1099, 1199, 1299, 1399, 1499, 1599]
MODELS = ["MULTICORPUS"]
for ep in EPOCHS:
    # MODELS.append(f"MULTICORPUS_swiglu_ep{ep}")
    # MODELS.append(f"MULTICORPUS_swiglu_muon_ep{ep}")
    MODELS.append(f"FT_swiglu_ep359_e{ep}")
# MODELS.append("FT_swiglu_ep359_e49")     # ← qui, nella STESSA cella
# MODELS.append("FT_swiglu_ep359_e99")
# MODELS.append("FT_swiglu_ep359_e149")
# MODELS.append("FT_swiglu_ep359_e199")
# MODELS.append("FT_swiglu_ep359_e299")
# MODELS.append("FT_swiglu_ep359_e399")
# # MODELS.append("MULTICORPUS_swiglu_muon_ep559")
# # MODELS.append("MULTICORPUS_swiglu_muon_ep619")
# # MODELS.append("MULTICORPUS_swiglu_muon_ep679")
# # MODELS.append("FT_swiglu_ep359_mrstft_e49")
# # MODELS.append("FT_swiglu_ep359_mrstft_e99")
# # MODELS.append("FT_swiglu_ep359_mrstft_e199")
# # MODELS.append("FT_swiglu_ep359_mrstft_e299")
# # MODELS.append("FT_swiglu_ep359_mrstft_e399")
# # MODELS.append("FT_swiglu_muon_ep499_e99")
# MODELS.append("FT_swiglu_ep359_e499")
# MODELS.append("FT_swiglu_ep359_e599")
# MODELS.append("FT_swiglu_ep359_e699")
# MODELS.append("FT_swiglu_ep359_e799")
# MODELS.append("FT_swiglu_ep359_e899")
# MODELS.append("FT_swiglu_ep359_e999")
# MODELS.append("FT_swiglu_ep359_e1099")
# MODELS.append("FT_swiglu_ep359_e1199")
# MODELS.append("FT_swiglu_ep359_e1299")
# MODELS.append("FT_swiglu_ep359_e1399")
# MODELS.append("FT_swiglu_ep359_e1499")

# Pretrain M/S (stereo-collapse fix): MAEB per capire se il LATENTE e' in plateau.
# La tabella recon restera' vuota per questi tag (le loro eval stanno in
# runs/ms_ablation_eval, non in runs/multicorpus_eval); qui servono le tabelle 3 e 4.
MODELS += ["MC_swiglu_ms_add_lrms_e319",
           "MC_swiglu_ms_add_lrms_e399",
           "MC_swiglu_ms_add_lrms_e499"]
MODELS = list(dict.fromkeys(MODELS))     # cintura di sicurezza



# 2.path functions
def _recon_subdir(m):
    if m == "MULTICORPUS":
        return "MULTICORPUS_baseline_latest"
    
    # Fallback temporaneo se MULTICORPUS_swiglu_ep299 è ancora salvato come epoch_299
    if m == "MULTICORPUS_swiglu_ep299":
        path_standard = Path(f"../runs/multicorpus_eval/recon_moisesdb_10s/{m}")
        if not path_standard.exists():
            return "epoch_299"
            
    return m

def _maeb_root(m):
    return Path(f"../runs/multicorpus_eval/maeb/maeb_{_recon_subdir(m)}")

def _maeb_label(m):
    return m

# 3. 
def recon_table_multicorpus(base_name, title):
    rows = []
    for m in MODELS:
        root = Path("../runs/multicorpus_eval")
        met = root / base_name / _recon_subdir(m) / "metrics"
        row = {"model": m}
        if met.exists():
            d = load_recon(met)
            for k in PERFILE: 
                arr = np.array(d.get(k, []))
                arr = arr[np.isfinite(arr)] # <--- FIX: ignora gli -inf dei brani muti di FMA
                row[k] = float(np.mean(arr)) if len(arr) > 0 else float("nan")
            for k in SCALAR:  row[k] = d.get(k, float("nan"))
            row["n_files"] = int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
            it = d.get("_infer", np.array([]))
            if len(it):
                row["infer_ms/file"] = float(np.median(it)) * 1e3
                at = d.get("_audio")
                row["RTF"] = float(np.median(it / np.clip(at, 1e-9, None))) if at is not None and len(at) == len(it) else float("nan")
            else:
                row["infer_ms/file"] = row["RTF"] = float("nan")
        rows.append(row)
    disp = PERFILE + SCALAR + ["n_files", "infer_ms/file", "RTF"]
    df = pd.DataFrame(rows).set_index("model").reindex(columns=disp)
    lower = (set(PERFILE + SCALAR) - HIGHER) | {"infer_ms/file", "RTF"}
    def hl(s):
        if s.name == "n_files": return [""] * len(s)
        fin = s[s.notna()]
        if fin.empty: return [""] * len(s)
        best = fin.min() if s.name in lower else fin.max()
        return ["font-weight:bold;color:#00cc66" if (pd.notna(v) and v == best) else "" for v in s]
    fmt = {**{c: "{:.4f}" for c in PERFILE + SCALAR}, "cdpam": "{:.3f}",
           "n_files": "{:.0f}", "infer_ms/file": "{:.1f}", "RTF": "{:.3f}"}
    display(Markdown(f"### {title}"))
    display(df.style.apply(hl).format(fmt, na_rep="—"))
    return df

display(Markdown("# MULTICORPUS: baseline vs swiglu vs swiglu-muon"))
recon_fma_mc  = recon_table_multicorpus("recon_fma", "1) FMA — reconstruction")
recon_mois_mc = recon_table_multicorpus("recon_moisesdb_10s", "2) MoisesDB-10s — reconstruction")
maeb_fma_mc   = maeb_table(CORE_FMA + MAEB_MUSIC, CORE_FMA, "3) FMA — MAEB probing")
maeb_mois_mc  = maeb_table(MOIS, MOIS, "4) MoisesDB — MAEB probing")


# MULTICORPUS: baseline vs swiglu vs swiglu-muon

### 1) FMA — reconstruction

,si_sdr,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann,n_files,infer_ms/file,RTF
model,,,,,,,,,,,,,
MULTICORPUS,2.3725,4.2863,1.0690,0.5140,0.083,0.9325,0.9359,0.2002,0.0330,—,11252,119.7,0.004
FT_swiglu_ep359_e99,3.4725,4.9634,1.0325,0.4871,0.078,0.9508,0.9554,0.1216,0.0223,—,11252,142.1,0.005
FT_swiglu_ep359_e199,3.4585,4.9506,1.0116,0.4789,0.078,0.9554,0.9598,0.1067,0.0177,—,11252,142.1,0.005
FT_swiglu_ep359_e299,3.4526,4.9404,1.0092,0.4793,0.078,0.9574,0.9617,0.0991,0.0158,—,11252,142.0,0.005
FT_swiglu_ep359_e359,—,—,—,—,—,—,—,—,—,—,—,—,—
FT_swiglu_ep359_e399,3.4522,4.9345,1.0147,0.4819,0.078,0.9585,0.9626,0.0973,0.0150,—,11252,142.0,0.005
FT_swiglu_ep359_e499,3.4500,4.9276,1.0148,0.4804,0.078,0.9586,0.9628,0.0931,0.0146,—,11247,142.2,0.005
FT_swiglu_ep359_e599,3.4446,4.9224,1.0066,0.4790,0.078,0.9584,0.9626,0.0914,0.0149,—,11252,142.0,0.005
FT_swiglu_ep359_e699,3.4452,4.9199,1.0014,0.4773,0.078,0.9590,0.9630,0.0894,0.0148,—,11252,142.3,0.005


### 2) MoisesDB-10s — reconstruction

,si_sdr,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,fad_mert,fad_gudgud,fad_pann,n_files,infer_ms/file,RTF
model,,,,,,,,,,,,,
MULTICORPUS,2.8478,4.3450,0.8469,0.3936,0.093,0.9493,0.9601,0.3288,0.0486,—,1998,44.6,0.004
FT_swiglu_ep359_e99,3.4231,4.7614,0.9082,0.3912,0.085,0.9633,0.9719,0.1809,0.0327,—,1998,52.3,0.005
FT_swiglu_ep359_e199,3.4187,4.7500,0.9055,0.3920,0.085,0.9674,0.9757,0.1618,0.0238,—,1998,56.6,0.006
FT_swiglu_ep359_e299,3.4148,4.7451,0.8957,0.3916,0.085,0.9694,0.9772,0.1494,0.0208,—,1998,52.4,0.005
FT_swiglu_ep359_e359,—,—,—,—,—,—,—,—,—,—,—,—,—
FT_swiglu_ep359_e399,3.4166,4.7463,0.8914,0.3908,0.085,0.9701,0.9773,0.1433,0.0205,—,1998,52.3,0.005
FT_swiglu_ep359_e499,3.4121,4.7394,0.8913,0.3923,0.084,0.9715,0.9784,0.1348,0.0182,—,1998,52.2,0.005
FT_swiglu_ep359_e599,3.4133,4.7394,0.8835,0.3907,0.084,0.9704,0.9777,0.1322,0.0198,—,1998,52.5,0.005
FT_swiglu_ep359_e699,3.4119,4.7398,0.8778,0.3903,0.084,0.9705,0.9776,0.1273,0.0200,—,1998,52.4,0.005


### 3) FMA — MAEB probing

,FMAGenreClassification,FMAGenreClustering,FMAArtistClustering,FMAArtistA2ARetrieval,FMAGenreAudioReranking,FMAArtistPairClassification,GTZANGenre,GTZANGenreClustering,MusicGenreClustering,GTZANAudioReranking,NSynth,JamAltArtistA2ARetrieval,AVG
MULTICORPUS,0.4967,0.3375,0.7044,0.5480,0.6128,0.9162,0.7620,—,0.3852,—,—,—,0.6026
FT_swiglu_ep359_e99,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e199,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e299,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e359,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e399,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e499,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e599,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e699,—,—,—,—,—,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e799,—,—,—,—,—,—,—,—,—,—,—,—,0.0000


### 4) MoisesDB — MAEB probing

,MoisesDBGenreClassification,MoisesDBGenreClustering,MoisesDBArtistClustering,MoisesDBArtistA2ARetrieval,MoisesDBGenreAudioReranking,MoisesDBArtistPairClassification,MoisesDBInstrumentClassification,AVG
MULTICORPUS,0.5911,0.2491,0.5528,0.5953,0.6365,0.7789,0.6858,0.5842
FT_swiglu_ep359_e99,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e199,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e299,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e359,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e399,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e499,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e599,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e699,—,—,—,—,—,—,—,0.0000
FT_swiglu_ep359_e799,—,—,—,—,—,—,—,0.0000


In [4]:
# =====================================================================
# FT CHECKPOINT RANKING — aggregate CLAP + 3×FAD over FMA + MoisesDB recon
#   Reuses recon_fma_mc / recon_mois_mc from the MULTICORPUS cell above.
#   For each FT checkpoint, sums every metric across the two reconstruction
#   sets (FMA + MoisesDB-10s), then ranks by a common-scale FAD score:
#       fad_score = fad_mert + 10·fad_gudgud + fad_pann   (lower = better)
#   Checkpoints missing any FAD on either set → score NaN → sink to bottom.
# =====================================================================
FAD_COLS  = ["fad_mert", "fad_gudgud", "fad_pann"]
CLAP_COLS = ["clap_audio", "clap_music"]
AGG_COLS  = CLAP_COLS + FAD_COLS

# checkpoints to rank = every FT model in the tables (drop the baseline)
ft_models = [m for m in MODELS if m != "MULTICORPUS"]

# sum FMA + MoisesDB per metric (index-aligned; NaN if missing on either set)
agg = (recon_fma_mc.reindex(ft_models)[AGG_COLS]
       + recon_mois_mc.reindex(ft_models)[AGG_COLS])

# common-scale ranking score (fad_gudgud rescaled ×10); lower is better
agg["fad_score"] = agg["fad_mert"] + agg["fad_gudgud"] * 10 + agg["fad_pann"]

# rank ascending; incomplete checkpoints (score NaN) go last
ranked = agg.sort_values("fad_score", na_position="last")

disp_cols = ["fad_score"] + FAD_COLS + CLAP_COLS
def _hl(s):
    fin = s[s.notna()]
    if fin.empty: return [""] * len(s)
    best = fin.max() if s.name in {"clap_audio", "clap_music"} else fin.min()  # clap↑, fad↓
    return ["font-weight:bold;color:#00cc66" if (pd.notna(v) and v == best) else "" for v in s]
fmt = {c: "{:.4f}" for c in disp_cols}

display(Markdown("### FT ranking — CLAP + FAD summed over FMA + MoisesDB (↓ fad_score = fad_mert + 10·fad_gudgud + fad_pann)"))
display(ranked[disp_cols].style.apply(_hl).format(fmt, na_rep="—"))

missing = ranked.index[ranked["fad_score"].isna()].tolist()
if missing:
    display(Markdown(f"*Esclusi dal ranking (FAD mancante su FMA o MoisesDB): {', '.join(missing)}*"))


### FT ranking — CLAP + FAD summed over FMA + MoisesDB (↓ fad_score = fad_mert + 10·fad_gudgud + fad_pann)

,fad_score,fad_mert,fad_gudgud,fad_pann,clap_audio,clap_music
model,,,,,,
FT_swiglu_ep359_e1299,2.5664,0.2076,0.0341,2.0182,1.9412,1.9301
FT_swiglu_ep359_e1499,2.5792,0.2057,0.0337,2.0361,1.9417,1.9307
FT_swiglu_ep359_e1199,2.5810,0.2079,0.0339,2.0338,1.9415,1.9299
FT_swiglu_ep359_e1399,2.5847,0.2058,0.0343,2.0359,1.9415,1.9297
FT_swiglu_ep359_e1099,2.6099,0.2069,0.0343,2.0603,1.9411,1.9300
FT_swiglu_ep359_e999,2.6170,0.2113,0.0345,2.0607,1.9410,1.9300
FT_swiglu_ep359_e899,2.6445,0.2128,0.0352,2.0800,1.9403,1.9290
FT_swiglu_ep359_e99,—,0.3025,0.0550,—,1.9273,1.9141
FT_swiglu_ep359_e199,—,0.2685,0.0414,—,1.9355,1.9228


*Esclusi dal ranking (FAD mancante su FMA o MoisesDB): FT_swiglu_ep359_e99, FT_swiglu_ep359_e199, FT_swiglu_ep359_e299, FT_swiglu_ep359_e359, FT_swiglu_ep359_e399, FT_swiglu_ep359_e499, FT_swiglu_ep359_e599, FT_swiglu_ep359_e699, FT_swiglu_ep359_e799*

In [ ]:
# =====================================================================
# ABLATION: CLAP vs MERT semantic-distill teacher (FMA-medium, LARGE2 recipe, 97 ep)
# SELF-CONTAINED: reads ONLY runs/med_ablation/. Two arms differ solely in the
# distillation teacher (CLAP global (B,512) vs MERT-v1-95M layer-4 framewise (B,768,T)).
# Tables: (2) MoisesDB-10s reconstruction (all 1998 chunks) + (4) MoisesDB 30s MAEB (7 tasks).
# Mirrors the loaders of the FINAL EVAL cell; MERT layer=4 for fad_mert (same as teacher).
# =====================================================================
import csv as _csv, json
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown

MA = Path("../runs/med_ablation")
# display_name -> (recon_stem under recon_moisesdb_10s, maeb root under maeb/)
ARMS = {
    "CLAP-distill": ("MED_LARGE2_clap_e96", MA / "maeb" / "maeb_clap"),
    "MERT-distill": ("MED_LARGE2_mert_e96", MA / "maeb" / "maeb_mert"),
}

# ---------- reconstruction loaders (mirror FINAL EVAL cell) ----------
def _col(path, col):
    out = []
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: out.append(float(r[col]))
            except (KeyError, ValueError, TypeError): pass
    return np.asarray(out, dtype=float)

def _scalar(path, col="score"):
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: return float(r[col])
            except (KeyError, ValueError, TypeError): return float("nan")
    return float("nan")

def load_recon(metrics: Path) -> dict:
    d = {}
    sp = metrics / "spectral.csv"
    if sp.exists():
        for k in ("si_sdr", "sdr", "stft_loss", "mel_loss"): d[k] = _col(sp, k)
    if (metrics / "cdpam.csv").exists(): d["cdpam"] = _col(metrics / "cdpam.csv", "cdpam")
    for fn, k in (("clap_music.csv", "clap_music"), ("clap_audio.csv", "clap_audio")):
        if (metrics / fn).exists(): d[k] = _col(metrics / fn, "cosine")
    for fn, k in (("fad_mert.csv", "fad_mert"), ("fad_gudgud.csv", "fad_gudgud"), ("fad_pann.csv", "fad_pann")):
        if (metrics / fn).exists(): d[k] = _scalar(metrics / fn, "score")
    return d

PERFILE = ["si_sdr", "sdr", "stft_loss", "mel_loss", "cdpam", "clap_music", "clap_audio"]
SCALAR  = ["fad_mert", "fad_gudgud", "fad_pann"]
HIGHER  = {"si_sdr", "sdr", "clap_music", "clap_audio"}   # else lower-is-better

def recon_table():
    rows = []
    for name, (stem, _) in ARMS.items():
        met = MA / "recon_moisesdb_10s" / stem / "metrics"
        row = {"model": name}
        if met.exists():
            d = load_recon(met)
            for k in PERFILE:
                arr = np.array(d.get(k, []), dtype=float); arr = arr[np.isfinite(arr)]
                row[k] = float(np.mean(arr)) if len(arr) else float("nan")
            for k in SCALAR: row[k] = d.get(k, float("nan"))
            row["n_files"] = int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
        else:
            for k in PERFILE + SCALAR: row[k] = float("nan")
            row["n_files"] = 0
        rows.append(row)
    disp = PERFILE + SCALAR + ["n_files"]
    df = pd.DataFrame(rows).set_index("model").reindex(columns=disp)
    lower = set(PERFILE + SCALAR) - HIGHER
    def hl(s):
        if s.name == "n_files": return [""] * len(s)
        fin = s[s.notna()]
        if fin.empty: return [""] * len(s)
        best = fin.min() if s.name in lower else fin.max()
        return ["font-weight:bold;color:#00cc66" if (pd.notna(v) and v == best) else "" for v in s]
    fmt = {**{c: "{:.4f}" for c in PERFILE + SCALAR}, "cdpam": "{:.3f}", "n_files": "{:.0f}"}
    display(Markdown("### 2) MoisesDB-10s — reconstruction (all 1998 chunks)"))
    display(df.style.apply(hl).format(fmt, na_rep="—"))
    return df

# ---------- MAEB loaders (mirror FINAL EVAL cell) ----------
def _main_score(path: Path):
    d = json.loads(path.read_text())
    for sv in d.get("scores", {}).values():
        if isinstance(sv, list):
            for s in sv:
                if isinstance(s, dict) and "main_score" in s: return float(s["main_score"])
        elif isinstance(sv, dict) and "main_score" in sv:
            return float(sv["main_score"])
    return None

def load_maeb(root: Path) -> dict:
    res = {}
    for local in root.glob("*/local"):
        for jp in local.glob("*.json"):
            if jp.stem in ("summary", "model_meta"): continue
            sc = _main_score(jp)
            if sc is not None: res[jp.stem] = sc
    return res

MOIS = ["MoisesDBGenreClassification", "MoisesDBGenreClustering", "MoisesDBArtistClustering",
        "MoisesDBArtistA2ARetrieval", "MoisesDBGenreAudioReranking",
        "MoisesDBArtistPairClassification", "MoisesDBInstrumentClassification"]

def maeb_table():
    data = {name: load_maeb(root) for name, (_, root) in ARMS.items()}
    rows = []
    for name in ARMS:
        r = [data[name].get(t) for t in MOIS]
        fin = [v for v in r if v is not None]
        r.append(sum(fin) / len(MOIS) if fin else float("nan"))
        rows.append(r)
    df = pd.DataFrame(rows, index=list(ARMS), columns=MOIS + ["AVG"])
    def hl(s):
        fin = s[s.notna()]
        if fin.empty: return [""] * len(s)
        best = fin.max()
        return ["font-weight:bold;color:#4C72B0" if (pd.notna(v) and v == best) else "" for v in s]
    display(Markdown("### 4) MoisesDB 30s — MAEB probing (7 tasks, AVG over all)"))
    display(df.style.apply(hl, axis=0).format("{:.4f}", na_rep="—"))
    return df

display(Markdown("# ABLATION — CLAP vs MERT semantic-distill teacher  \n"
                 "_FMA-medium, LARGE2 recipe, 97 ep. Self-contained; reads `runs/med_ablation/` only._"))
abl_recon = recon_table()
abl_maeb  = maeb_table()


# ABLATION — CLAP vs MERT semantic-distill teacher  
_FMA-medium, LARGE2 recipe, 97 ep. Self-contained; reads `runs/med_ablation/` only._

### 2) MoisesDB-10s — reconstruction (all 1998 chunks)

,si_sdr,sdr,stft_loss,mel_loss,cdpam,clap_music,clap_audio,mert_cos,fad_mert,fad_gudgud,fad_pann,n_files
model,,,,,,,,,,,,
CLAP-distill,1.6771,3.6656,1.0466,0.4984,0.109,0.9333,0.9436,0.9057,0.9052,0.0630,3.0272,1998
MERT-distill,2.2017,4.0558,1.0090,0.4729,0.113,0.9275,0.9377,0.9044,1.0311,0.0752,3.3815,1998


### 4) MoisesDB 30s — MAEB probing (7 tasks, AVG over all)

,MoisesDBGenreClassification,MoisesDBGenreClustering,MoisesDBArtistClustering,MoisesDBArtistA2ARetrieval,MoisesDBGenreAudioReranking,MoisesDBArtistPairClassification,MoisesDBInstrumentClassification,AVG
CLAP-distill,0.5467,0.1842,0.5348,0.5710,0.6228,0.7807,0.6364,0.5538
MERT-distill,0.4222,0.1577,0.5096,0.5458,0.5693,0.7417,0.5598,0.5009


In [1]:
# =====================================================================
# STEREO M/S ABLATION — ms_add vs ms_replace (STEREO_COLLAPSE fix)
# SELF-CONTAINED. Reads runs/ms_ablation_eval/{recon_fma,recon_moisesdb_10s}/<TAG>/metrics.
# TAGs = checkpoint symlink stems: MC_swiglu_ms_replace_e99 / MC_swiglu_ms_add_e99
# (swin_real_swiglu_xsa = SAGE arch, epoch 99, MoisesDB-10s; run 300ep in corso).
# Mostra le metriche recon standard + le 3 metriche stereo (ms_metrics.csv):
#   width_bias  (→0; <0 squash, >0 over-widening)   [zero-best]
#   d_width     (→0, errore d'immagine assoluto)     [lower-best]
#   sisdr_s     (dB, SI-SDR del canale Side; →su)     [higher-best]
#   sisdr_m     (dB, SI-SDR del canale Mid; →su)      [higher-best, guardrail Mid]
# Target dalla diagnosi (12 tracce): SAME width_bias≈+0.01 d_width≈0.057 sisdr_s≈+2.6;
#                                     SAGE-baseline width_bias≈-0.14 d_width≈0.145 sisdr_s≈-12.8.
# =====================================================================
import csv as _csv
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown

MS_ROOT = Path("../runs/ms_ablation_eval")   # namespace dedicato agli ablation M/S


# Solo le due run stereo-collapse swiglu @ epoch 99 (SAGE arch, eval MoisesDB-10s).
MS_MODELS = [
            # "MC_500ep_e99",          # RIFERIMENTO: arch BASELINE senza swiglu (vanilla assoluto)
            #  "MC_swiglu_500ep_e319",  # RIFERIMENTO: swiglu vanilla (no mrstft_sd, no fold_lrms)
            #  "MC_swiglu_ms_replace_e99",  "MC_swiglu_ms_add_e99",
            #  "MC_swiglu_ms_replace_e199", "MC_swiglu_ms_add_e199",
            #  "MC_swiglu_ms_replace_e239", "MC_swiglu_ms_add_e239",
            #  "MC_swiglu_lrms_disc_e99",   "MC_swiglu_ms_add_lrms_e99",
            #  "MC_swiglu_lrms_disc_e219",  "MC_swiglu_ms_add_lrms_e219",
            #  "MC_swiglu_ms_add_lrms_e319",  "MC_swiglu_lrms_mse_side_e239",
            #  "MC_swiglu_ms_add_lrms_e359",  "MC_swiglu_ms_add_lrms_e399",
             "MC_swiglu_ms_add_lrms_e499",
             # FT (decoder-finetune) dal pretrain e499: encoder CONGELATO,
             # quindi lo stereo del latente e' ereditato e non riallenabile.
             "FT_A_mrstft_sd_e99", "FT_B_no_mrstft_sd_e99", "FT_A_mrstft_sd_e199",
             "FT_B_no_mrstft_sd_e199", "FT_A_mrstft_sd_e299", "FT_B_no_mrstft_sd_e299",
             "FT_A_mrstft_sd_e399", "FT_B_no_mrstft_sd_e399",
             "FT_A_mrstft_sd_e499", "FT_B_no_mrstft_sd_e499", 
             "FT_A_mrstft_sd_e599", "FT_B_no_mrstft_sd_e599",
             "FT_A_mrstft_sd_e699", "FT_B_no_mrstft_sd_e699",
             "FT_A_mrstft_sd_e799", "FT_B_no_mrstft_sd_e799",
             "FT_A_mrstft_sd_e899", "FT_A_mrstft_sd_e992",
             # Stesso checkpoint di e992, ma con il fix variable-length tri2 attivo
             # (default di progetto dal 2026-09-22). A pesi identici, la differenza
             # fra le due righe e' l'effetto del solo fix. Vedi VARLEN_SEAMS.md 8.4.
             "FT_A_mrstft_sd_e992_tri2","FT_A_mrstft_sd_e1092_tri2"]

def _ms_col(path, col):
    out=[]
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: out.append(float(r[col]))
            except (KeyError, ValueError, TypeError): pass
    return np.asarray(out, dtype=float)

def _ms_scalar(path, col="score"):
    with open(path, newline="") as f:
        for r in _csv.DictReader(f):
            try: return float(r[col])
            except (KeyError, ValueError, TypeError): return float("nan")
    return float("nan")

def load_recon_ms(metrics: Path) -> dict:
    d={}
    sp=metrics/"spectral.csv"
    if sp.exists():
        for k in ("sdr","stft_loss"): d[k]=_ms_col(sp,k)
    if (metrics/"cdpam.csv").exists(): d["cdpam"]=_ms_col(metrics/"cdpam.csv","cdpam")
    for fn,k in (("clap_music.csv","clap_music"),("clap_audio.csv","clap_audio")):
        if (metrics/fn).exists(): d[k]=_ms_col(metrics/fn,"cosine")
    for fn,k in (("fad_mert.csv","fad_mert"),("fad_gudgud.csv","fad_gudgud"),("fad_pann.csv","fad_pann")):
        if (metrics/fn).exists(): d[k]=_ms_scalar(metrics/fn,"score")
    ms=metrics/"ms_metrics.csv"                                   # ← nuove metriche stereo
    if ms.exists():
        for k in ("width_bias","d_width","sisdr_s","sisdr_m"): d[k]=_ms_col(ms,k)
    return d

MS_PERFILE = ["sdr","stft_loss","cdpam","clap_music","clap_audio","width_bias","d_width","sisdr_s","sisdr_m"]
MS_SCALAR  = ["fad_mert","fad_gudgud","fad_pann"]
MS_HIGHER  = {"si_sdr","clap_music","clap_audio","sisdr_s","sisdr_m"}  # higher-better
MS_ZEROBEST= {"width_bias"}                                       # closest-to-0 best
MS_STEREO  = ["width_bias","d_width","sisdr_s","sisdr_m"]         # per evidenziarle in grassetto colonna

def recon_table_ms(base_name, title):
    rows=[]
    for m in MS_MODELS:
        met=MS_ROOT/base_name/m/"metrics"
        row={"model":m}
        if met.exists():
            d=load_recon_ms(met)
            for k in MS_PERFILE:
                arr=np.array(d.get(k,[]), dtype=float)
                arr=arr[np.isfinite(arr)]                          # ignora -inf (side muto) / NaN
                row[k]=float(np.mean(arr)) if len(arr)>0 else float("nan")
            for k in MS_SCALAR: row[k]=d.get(k,float("nan"))
            row["n_files"]=int(max((len(v) for v in d.values() if isinstance(v, np.ndarray)), default=0))
        else:
            for k in MS_PERFILE+MS_SCALAR: row[k]=float("nan")
            row["n_files"]=0
        rows.append(row)
    disp=MS_PERFILE+MS_SCALAR+["n_files"]
    df=pd.DataFrame(rows).set_index("model").reindex(columns=disp)
    df=df.dropna(axis=0, how="all", subset=MS_PERFILE+MS_SCALAR)   # nascondi tag non ancora valutati
    lower=(set(MS_PERFILE+MS_SCALAR)-MS_HIGHER-MS_ZEROBEST)
    def hl(s):
        if s.name=="n_files": return [""]*len(s)
        fin=s[s.notna()]
        if fin.empty: return [""]*len(s)
        if s.name in MS_ZEROBEST: best=fin.index[fin.abs().argmin()]      # più vicino a 0
        elif s.name in MS_HIGHER: best=fin.idxmax()
        else:                     best=fin.idxmin()
        return ["font-weight:bold;color:#00cc66" if i==best else "" for i in s.index]
    fmt={**{c:"{:.4f}" for c in MS_PERFILE+MS_SCALAR}, "cdpam":"{:.5f}",
         "sisdr_s":"{:+.2f}", "sisdr_m":"{:+.2f}", "si_sdr":"{:.2f}", "width_bias":"{:+.4f}", "n_files":"{:.0f}"}
    # colora le 3 colonne stereo per distinguerle
    def col_bg(c): return ["background-color:#12263a"]*len(df) if c.name in MS_STEREO else [""]*len(df)
    display(Markdown(f"### {title}"))
    display(df.style.apply(hl).apply(col_bg, axis=0).format(fmt, na_rep="—"))
    return df

display(Markdown("# STEREO M/S ABLATION — ms_add vs ms_replace"))
display(Markdown("**Direzione giusta**: `width_bias`→0, `d_width`↓, `sisdr_s`↑, "
                 "SENZA regressione su `fad_*`/`mel_loss` (guardrail Mid, soprattutto per `replace`)."))
ms_fma  = recon_table_ms("recon_fma",           "FMA — recon + stereo M/S")
ms_mois = recon_table_ms("recon_moisesdb_10s",  "MoisesDB-10s — recon + stereo M/S")


# STEREO M/S ABLATION — ms_add vs ms_replace

**Direzione giusta**: `width_bias`→0, `d_width`↓, `sisdr_s`↑, SENZA regressione su `fad_*`/`mel_loss` (guardrail Mid, soprattutto per `replace`).

### FMA — recon + stereo M/S

,sdr,stft_loss,cdpam,clap_music,clap_audio,width_bias,d_width,sisdr_s,sisdr_m,fad_mert,fad_gudgud,fad_pann,n_files
model,,,,,,,,,,,,,


### MoisesDB-10s — recon + stereo M/S

,sdr,stft_loss,cdpam,clap_music,clap_audio,width_bias,d_width,sisdr_s,sisdr_m,fad_mert,fad_gudgud,fad_pann,n_files
model,,,,,,,,,,,,,
MC_swiglu_ms_add_lrms_e499,4.6959,0.8667,0.06428,0.9502,0.9576,+0.0024,0.0604,-5.09,+4.84,0.3047,0.0539,1.6798,1998
FT_A_mrstft_sd_e99,4.7169,0.8887,0.06206,0.9476,0.9547,+0.0030,0.0604,-5.04,+4.84,0.3458,0.0591,1.9173,1998
FT_B_no_mrstft_sd_e99,4.9202,0.9104,0.07086,0.9503,0.9739,-0.0128,0.0595,-5.18,+4.70,0.3242,0.0208,1.4135,1998
FT_A_mrstft_sd_e199,4.7224,0.8798,0.06232,0.9551,0.9618,+0.0032,0.0604,-5.05,+4.84,0.3034,0.0445,1.6227,1998
FT_B_no_mrstft_sd_e199,4.9199,0.8925,0.06598,0.9530,0.9754,-0.0147,0.0593,-5.18,+4.67,0.2841,0.0196,1.2788,1998
FT_A_mrstft_sd_e299,4.7309,0.8815,0.06203,0.9594,0.9655,+0.0031,0.0603,-5.04,+4.84,0.2806,0.0375,1.4987,1998
FT_B_no_mrstft_sd_e299,4.9134,0.8839,0.06692,0.9563,0.9762,-0.0154,0.0592,-5.19,+4.66,0.2596,0.0190,1.1913,1998
FT_A_mrstft_sd_e399,4.7313,0.8740,0.06225,0.9618,0.9674,+0.0032,0.0603,-5.04,+4.84,0.2760,0.0336,1.4685,1998
FT_B_no_mrstft_sd_e399,4.9165,0.8800,0.06858,0.9591,0.9768,-0.0159,0.0592,-5.20,+4.65,0.2474,0.0186,1.1231,1998


In [4]:
# =====================================================================
# FAD & CDPAM — MID vs SIDE vs LRDIFF/LRSUM, tabella unica
# SELF-CONTAINED. Mette a confronto lo stesso set di 1998 chunk MoisesDB-10s
# valutato su quattro "canali":
#   MID    = wav.mean(0)      (il downmix mono su cui sono definite TUTTE le FAD
#                              storiche: e' invariante a qualsiasi guadagno Side)
#   SIDE   = (L - R) / 2      (l'informazione stereo, invisibile alle FAD Mid)
#   LRDIFF = (emb(L) - emb(R)) / 2   (sperimentale, evaluation/stereo_diagnosis/lrdiff/)
#   LRSUM  = (emb(L) + emb(R)) / 2   (controllo: quanto emb e' non lineare vs MID)
# Le reference stats di ogni canale hanno nomi DISTINTI (<model>, -side, -lrdiff,
# -lrsum): incrociarli e' un errore silenzioso che gonfia la FAD ~50x.
# ⚠️ I VALORI NON SONO CONFRONTABILI FRA CANALI (spazi diversi): solo gli ordinamenti.
#    LRDIFF dipende anche dal Mid (probe: runs/lrdiff_eval/probe_moisesdb_10s/).
#
# Celle sorgente:
#   MID    SAGE -> runs/ms_ablation_eval/recon_moisesdb_10s/<TAG>/metrics
#   MID    SOTA -> runs/eval_sota_10s/<model>/metrics          (fad_pann NON calcolata)
#   SIDE   SAGE -> runs/side_eval/recon_moisesdb_10s_side/<TAG>/metrics
#   SIDE   SOTA -> runs/side_eval/eval_sota_10s_side/<model>/metrics
#   LRDIFF SAGE -> runs/lrdiff_eval/recon_moisesdb_10s_lrdiff/<TAG>/metrics  (fad_*_{lrdiff,lrsum}.csv)
#   LRDIFF SOTA -> runs/lrdiff_eval/eval_sota_10s_lrdiff/<model>/metrics
# Le caselle senza file su disco sono "n.m." (non misurato), mai 0 e mai stimate.
# =====================================================================
import csv as _csv
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown

R           = Path("../runs")
MID_SAGE    = R / "ms_ablation_eval" / "recon_moisesdb_10s"
MID_SOTA    = R / "eval_sota_10s"
SIDE_SAGE   = R / "side_eval" / "recon_moisesdb_10s_side"
SIDE_SOTA   = R / "side_eval" / "eval_sota_10s_side"
LRDIFF_SAGE = R / "lrdiff_eval" / "recon_moisesdb_10s_lrdiff"
LRDIFF_SOTA = R / "lrdiff_eval" / "eval_sota_10s_lrdiff"

def _m(p: Path) -> Path:
    return p / "metrics"                      # i CSV stanno sempre in <run>/metrics/

# nome visualizzato -> (run dir MID, run dir SIDE, run dir LRDIFF/LRSUM)
def _sage(tag: str): return (MID_SAGE / tag, SIDE_SAGE / tag, LRDIFF_SAGE / tag)
def _sota(key: str): return (MID_SOTA / key, SIDE_SOTA / key, LRDIFF_SOTA / key)

CH_MODELS = {
    "SAGE pretrain e499":   _sage("MC_swiglu_ms_add_lrms_e499"),
    "FT_A  mrstft_sd e399": _sage("FT_A_mrstft_sd_e399"),
    "FT_B  no_mrstft  e399":_sage("FT_B_no_mrstft_sd_e399"),
    "SAME (SOTA)":          _sota("same"),
    "SAO-VAE (SOTA)":       _sota("sao-vae"),
}

def _ch_scalar(metrics: Path, fn: str, col: str = "score") -> float:
    """FAD: uno scalare per run. Assente su disco -> NaN (= 'non misurato')."""
    p = metrics / fn
    if not p.exists(): return float("nan")
    with open(p, newline="") as f:
        for r in _csv.DictReader(f):
            try: return float(r[col])
            except (KeyError, ValueError, TypeError): return float("nan")
    return float("nan")

def _ch_col(metrics: Path, fn: str, col: str) -> np.ndarray:
    p = metrics / fn
    if not p.exists(): return np.array([], dtype=float)
    out = []
    with open(p, newline="") as f:
        for r in _csv.DictReader(f):
            try: out.append(float(r[col]))
            except (KeyError, ValueError, TypeError): pass
    a = np.asarray(out, dtype=float)
    return a[np.isfinite(a)]

CH_FADS  = [("fad_mert", "fad_mert"), ("fad_gudgud", "fad_gudgud"), ("fad_pann", "fad_pann")]
CHANNELS = ("mid", "side", "lrdiff", "lrsum")

rows = []
for name, (mid_run, side_run, lr_run) in CH_MODELS.items():
    mid, side, lr = _m(mid_run), _m(side_run), _m(lr_run)
    row = {"model": name}
    for key, stem in CH_FADS:
        row[f"{key}_mid"]    = _ch_scalar(mid,  f"{stem}.csv")
        row[f"{key}_side"]   = _ch_scalar(side, f"{stem}.csv")
        row[f"{key}_lrdiff"] = _ch_scalar(lr,   f"{stem}_lrdiff.csv")
        row[f"{key}_lrsum"]  = _ch_scalar(lr,   f"{stem}_lrsum.csv")
    cm = _ch_col(mid,  "cdpam.csv", "cdpam")
    cs = _ch_col(side, "cdpam.csv", "cdpam")
    row["cdpam_mid"]  = float(cm.mean()) if len(cm) else float("nan")
    row["cdpam_side"] = float(cs.mean()) if len(cs) else float("nan")
    row["n_mid"], row["n_side"] = len(cm), len(cs)
    row["n_lrdiff"] = _ch_scalar(lr, "fad_mert_lrdiff.csv", col="n_files")   # file usati nella FAD
    rows.append(row)

CH_COLS = [f"{k}_{c}" for k, _ in CH_FADS for c in CHANNELS] + ["cdpam_mid", "cdpam_side"]
N_COLS  = ["n_mid", "n_side", "n_lrdiff"]
ch_df = pd.DataFrame(rows).set_index("model").reindex(columns=CH_COLS + N_COLS)

def _ch_hl(s):                                   # tutte lower-is-better
    if s.name.startswith("n_"): return [""] * len(s)
    fin = s[s.notna()]
    if fin.empty: return [""] * len(s)
    best = fin.idxmin()
    return ["font-weight:bold;color:#00cc66" if i == best else "" for i in s.index]

_BG = {"_side": "#12263a", "_lrdiff": "#2e1a3a", "_lrsum": "#1f2a1f"}
def _ch_bg(c):                                   # SIDE / LRDIFF / LRSUM su fondi diversi
    color = next((v for k, v in _BG.items() if c.name.endswith(k)), None)
    return [f"background-color:{color}" if color else ""] * len(ch_df)

_fmt = {**{c: "{:.4f}" for c in CH_COLS}, **{c: "{:.0f}" for c in N_COLS}}
display(Markdown("### FAD e CDPAM — MID vs SIDE vs LRDIFF/LRSUM (MoisesDB-10s, 1998 chunk)\n"
                 "Valori **non confrontabili fra canali**: leggere gli ordinamenti per colonna."))
display(ch_df.style.apply(_ch_hl).apply(_ch_bg, axis=0).format(_fmt, na_rep="n.m."))

# ── test appaiato A vs B sul CDPAM: e' l'unica metrica per-file, quindi l'unica
#    che dia un intervallo di confidenza. Le FAD (mid/side/lrdiff/lrsum) sono scalari
#    di run (una distanza fra distribuzioni) e non hanno varianza per-file.
def _paired(fa: Path, fb: Path, label: str, n_boot: int = 5000, seed: int = 0):
    def _byfile(m):
        p = m / "cdpam.csv"
        if not p.exists(): return {}
        with open(p, newline="") as f:
            return {r["file"]: float(r["cdpam"]) for r in _csv.DictReader(f)
                    if r.get("cdpam") not in (None, "") and np.isfinite(float(r["cdpam"]))}
    A, B = _byfile(fa), _byfile(fb)
    keys = sorted(set(A) & set(B))
    if not keys:
        print(f"[{label}] non misurato — manca almeno uno dei due cdpam.csv"); return
    d = np.array([B[k] - A[k] for k in keys])          # >0 => B peggio di A
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    bs  = d[idx].mean(axis=1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"[{label}] n={len(keys)}  Δ(B−A) = {d.mean():+.5f}  CI95 = [{lo:+.5f}, {hi:+.5f}]  "
          f"B peggiore su {100*(d>0).mean():.1f}% dei file")

### FAD e CDPAM — MID vs SIDE vs LRDIFF/LRSUM (MoisesDB-10s, 1998 chunk)
Valori **non confrontabili fra canali**: leggere gli ordinamenti per colonna.

,fad_mert_mid,fad_mert_side,fad_mert_lrdiff,fad_mert_lrsum,fad_gudgud_mid,fad_gudgud_side,fad_gudgud_lrdiff,fad_gudgud_lrsum,fad_pann_mid,fad_pann_side,fad_pann_lrdiff,fad_pann_lrsum,cdpam_mid,cdpam_side,n_mid,n_side,n_lrdiff
model,,,,,,,,,,,,,,,,,
SAGE pretrain e499,0.3047,0.6909,0.7493,0.3473,0.0539,0.0657,0.0039,0.1464,1.6798,4.2019,0.3228,1.9506,0.0643,0.0896,1998,1998,1998
FT_A mrstft_sd e399,0.2760,0.6683,0.7174,0.3171,0.0336,0.0715,0.0035,0.1041,1.4685,4.2247,0.3093,1.6505,0.0623,0.0908,1998,1998,1998
FT_B no_mrstft e399,0.2474,0.9892,0.9477,0.3215,0.0186,0.1177,0.0041,0.0802,1.1231,4.3987,0.3585,1.5280,0.0686,0.1582,1998,1998,1998
SAME (SOTA),0.2652,0.4128,0.0453,0.2305,0.0330,0.0568,0.0016,0.0319,n.m.,2.6855,0.1654,1.1010,0.0554,0.0797,1998,1998,1998
SAO-VAE (SOTA),0.5101,0.3957,0.0691,0.4219,0.1181,0.1411,0.0026,0.1125,n.m.,3.4413,0.2180,2.2511,0.0778,0.0901,1998,1998,1998
